Regresión Lineal Multivariante

In [1]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (20, 8), 'axes.titlesize': 22, 'axes.labelsize': 18,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 16,
    'lines.linewidth': 2.5
})

# --- PARÁMETROS ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
N_LAGS_AQI = 120
N_LAGS_EXO = 120
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_CV_SPLITS = 5
HISTORY_LEN_FOR_PLOT = 300
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Finetuned',
    'MOMENT Pretrained', 'Random Forest', 'SARIMA'
]

# --- RUTAS ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'regresionLineal_full_FT'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')
for d in [PREDICTION_METRICS_OUTPUT_DIR, PREDICTION_PLOTS_OUTPUT_DIR]: os.makedirs(d, exist_ok=True)

# --- FUNCIONES AUXILIARES ---
def create_multivariate_lagged_features(df, target_col, exo_cols, time_cols, n_lags_target, n_lags_exo, horizon):
    features = []
    for i in range(horizon, n_lags_target + horizon): features.append(df[target_col].shift(i).rename(f'{target_col}_lag{i}'))
    for exo_col in exo_cols:
        if exo_col == target_col: continue
        for i in range(horizon, n_lags_exo + horizon): features.append(df[exo_col].shift(i).rename(f'{exo_col}_lag{i}'))
    for tf_col in time_cols:
        if tf_col in df.columns: features.append(df[tf_col])
    return pd.concat(features, axis=1)

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

def generate_multi_step_forecasting_plots(df_imputed, df_ground_truth, model, X_all, original_nan_indices, plot_params):
    max_lags, horizon = max(N_LAGS_AQI, N_LAGS_EXO), plot_params['horizon']
    possible_points = range(max_lags + horizon + HISTORY_LEN_FOR_PLOT, len(df_imputed) - horizon - 1)
    valid_points = [p for p in possible_points if not any(idx in original_nan_indices for idx in range(p, p+horizon)) and any(idx in original_nan_indices for idx in range(p-max_lags-horizon, p))] if original_nan_indices else list(possible_points)
    if not valid_points: return
    split_point = random.choice(valid_points)
    try:
        plt.figure(figsize=(20, 8))
        history, future_gt = df_imputed.iloc[:split_point], df_ground_truth.iloc[split_point:split_point+horizon]
        X_forecast = X_all.loc[future_gt.index].dropna()
        if X_forecast.empty: plt.close(); return
        predictions = model.predict(X_forecast)
        plot_history = history.iloc[-HISTORY_LEN_FOR_PLOT:]
        plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
        plt.plot(future_gt.index, future_gt[TARGET_VARIABLE], label='Ground Truth', color='red')
        plt.plot(X_forecast.index, predictions, label='Forecast', color='green', linestyle='--')
        plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
        plt.title(f"Linear Regressor Forecast in {plot_params['city']} for {horizon}-Hour Horizon\nImputation: {plot_params['method']} | Scenario: {plot_params['scenario']}")
        plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        scenario_safe = plot_params['scenario'].replace('/', '_').replace('\\', '_')
        plot_save_dir = os.path.join(plot_params['output_dir'], plot_params['method'], plot_params['city'].replace(" ", "_"), scenario_safe)
        os.makedirs(plot_save_dir, exist_ok=True)
        plot_filename = f"forecast_example_{plot_params['city'].replace(' ', '_')}_{scenario_safe}_{plot_params['rep_id']}_h{horizon}.png"
        plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
        plt.close()
    except Exception:
        plt.close()

def train_predict_eval_multi_horizon(df_imputed, df_ground_truth, original_nan_indices, target_col, exo_cols, time_cols, n_lags_target, n_lags_exo, horizons, n_cv_splits, plot_params=None):
    results = []
    max_lags = max(n_lags_target, n_lags_exo)
    for horizon in tqdm(horizons, desc="    Horizons", leave=False):
        X = create_multivariate_lagged_features(df_imputed, target_col, exo_cols, time_cols, n_lags_target, n_lags_exo, horizon)
        y = df_ground_truth[target_col]
        full_data = pd.concat([X, y], axis=1).dropna()
        X_clean, y_clean = full_data.drop(columns=[target_col]), full_data[target_col]
        
        if X_clean.empty or len(X_clean) < n_cv_splits * 2:
            results.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan}); continue
        
        model = LinearRegression().fit(X_clean, y_clean)
        if plot_params and horizon == max(horizons):
            plot_params['horizon'] = horizon
            generate_multi_step_forecasting_plots(df_imputed, df_ground_truth, model, X, original_nan_indices, plot_params)
            
        tscv = TimeSeriesSplit(n_splits=n_cv_splits)
        metrics = {'maes': [], 'rmses': [], 'mases': []}
        
        for _, (train_idx, test_idx) in enumerate(tscv.split(X_clean)):
            y_train_for_mase = y_clean.iloc[train_idx]
            test_indices_from_split = X_clean.iloc[test_idx].index
            
            # --- LÍNEA CORREGIDA ---
            impact_indices = [idx for idx in test_indices_from_split if any(h in original_nan_indices for h in range(idx - (horizon + max_lags), idx))] if original_nan_indices else test_indices_from_split.tolist()
            
            if not impact_indices: continue
            X_eval, y_eval_true = X_clean.loc[impact_indices], y_clean.loc[impact_indices]
            if X_eval.empty: continue
            
            preds = model.predict(X_eval)
            metrics['maes'].append(mean_absolute_error(y_eval_true, preds))
            metrics['rmses'].append(np.sqrt(mean_squared_error(y_eval_true, preds)))
            mase = calculate_mase(y_eval_true, preds, y_train_for_mase)
            if not np.isnan(mase): metrics['mases'].append(mase)
        
        results.append({'Horizon': horizon, 'MAE': np.mean(metrics['maes']), 'RMSE': np.mean(metrics['rmses']), 'MASE': np.mean(metrics['mases'])})
    return results

def process_lr_task(task_params):
    method_name, city_name, original_data_path = task_params['method_name'], task_params['city_name'], task_params['original_data_path']
    df_original_full = pd.read_csv(original_data_path)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'].astype(str).str[:19])
    if 'CO2' in df_original_full.columns: df_original_full = df_original_full.drop(columns=['CO2'])
    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS: df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce').ffill().bfill()
    df_original_full['Hour'], df_original_full['DayOfWeek'], df_original_full['Month'] = df_original_full['Date'].dt.hour, df_original_full['Date'].dt.dayofweek, df_original_full['Date'].dt.month
    
    if method_name == 'Original_Data_Baseline':
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_input.isnull().values.any(): return []
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
        horizon_metrics = train_predict_eval_multi_horizon(df_city_input, df_city_input, set(), TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_params)
        return [{'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]
    
    scenario_folder, city_folder = task_params['scenario_folder'], city_name.replace(" ", "_")
    df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
    scenario_path = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs', city_folder, scenario_folder)
    try: rep_file_path = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[0]
    except IndexError: return []
    try:
        df_rep = pd.read_csv(rep_file_path)
        df_rep['Date'] = pd.to_datetime(df_rep['Date'].astype(str).str[:19])
        if not all(col in df_rep.columns for col in TIME_FEATURES):
            df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
        if df_rep.isnull().values.any(): df_rep = df_rep.ffill().bfill()
        if df_rep.isnull().values.any(): return []
    except Exception: return []
    
    rep_file_id = os.path.basename(rep_file_path)
    plot_params = {'city': city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
    original_nan_indices = set()
    try:
        nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id.split('_imputed')[0]}_with_nans.csv" if '_imputed' in rep_file_id else f"{city_folder}_{scenario_folder}_{rep_file_id}_with_nans.csv"
        df_with_nans = pd.read_csv(os.path.join(SIMULATED_DATA_ROOT_DIR, city_folder, scenario_folder, nan_filename))
        original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
    except FileNotFoundError: pass
    
    horizon_metrics = train_predict_eval_multi_horizon(df_rep, df_city_ground_truth, original_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_params)
    return [{'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

if __name__ == '__main__':
    script_start_time = time.time()
    if not os.path.exists(ORIGINAL_DATA_FILE): exit()
    cities_to_process = pd.read_csv(ORIGINAL_DATA_FILE, usecols=['City'])['City'].unique()
    tasks = []
    for city_name in cities_to_process:
        tasks.append(delayed(process_lr_task)({'method_name': 'Original_Data_Baseline', 'city_name': city_name, 'original_data_path': ORIGINAL_DATA_FILE}))
    for method_name in METHODS_TO_PROCESS:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue
        for city_name in cities_to_process:
            city_folder = city_name.replace(" ", "_")
            city_path = os.path.join(imputed_csvs_root, city_folder)
            if not os.path.isdir(city_path): continue
            for scenario_folder in os.listdir(city_path):
                if os.path.isdir(os.path.join(city_path, scenario_folder)):
                    tasks.append(delayed(process_lr_task)({'method_name': method_name, 'city_name': city_name, 'scenario_folder': scenario_folder, 'original_data_path': ORIGINAL_DATA_FILE}))
    
    print(f"\nSe han generado {len(tasks)} tareas para procesar en paralelo.")
    results_list = Parallel(n_jobs=-1, backend="loky")(tqdm(tasks, desc="Procesando Tareas LR"))
    
    print("\nProcesamiento completado. Generando resúmenes...")
    all_metrics = [item for sublist in results_list for item in sublist if sublist]
    if all_metrics:
        df_all_metrics = pd.DataFrame(all_metrics)
        detailed_output_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.to_csv(detailed_output_path, index=False)
        print(f"\n[OK] Métricas detalladas guardadas en: {detailed_output_path}")
        
        df_summary = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index()
        df_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_and_scenario_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print(f"[OK] Resumen por ciudad y escenario guardado.")
        
        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        df_global_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print(f"[OK] Resumen global guardado.")
        
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        df_avg_over_horizons.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print(f"[OK] Resumen final guardado.")
        
        print("\n--- Resumen Final (promedio de todo) ---\n", df_avg_over_horizons)
    else:
        print("\nNo se generaron resultados.")
        
    print(f"\nScript completado. Tiempo total: {(time.time() - script_start_time)/60:.2f} minutos.")


Se han generado 222 tareas para procesar en paralelo.


Procesando Tareas LR: 100%|██████████| 222/222 [08:09<00:00,  2.20s/it]



Procesamiento completado. Generando resúmenes...

[OK] Métricas detalladas guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\predicciones\regresionLineal_full_FT\metricas\ALL_prediction_metrics_detailed_regresionLineal_full_FT.csv
[OK] Resumen por ciudad y escenario guardado.
[OK] Resumen global guardado.
[OK] Resumen final guardado.

--- Resumen Final (promedio de todo) ---
         Imputation_Method  Final_Avg_MAE  Final_Avg_RMSE  Final_Avg_MASE
0                LSTM-VAE       6.798832        8.921563        6.028944
1    Linear Interpolation       6.563524        8.598192        5.808067
2        MOMENT Finetuned       6.534631        8.582927        5.791861
3       MOMENT Pretrained       6.582524        8.643562        5.833601
4  Original_Data_Baseline       6.460670        8.481927        5.735788
5           Random Forest       6.568019        8.613263        5.816801
6                  SARIMA       6.683835        8.780195        5.933404

Script co

SARIMAX PARALELIZADO

In [1]:
import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de Estilo para Gráficas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['lines.linewidth'] = 2.5

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]

# --- PARÁMETROS DE EJECUCIÓN ---
N_CV_SPLITS = 2
HISTORY_LEN_FOR_PLOT = 300
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Finetuned',
    'MOMENT Pretrained', 'Random Forest', 'SARIMA'
]

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', 'SARIMA', 'model_orders')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'SARIMAX_full_FT'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')

os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_OUTPUT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

def generate_sarimax_forecasting_plots(df_full_imputed, df_full_ground_truth, model_results, original_nan_indices, city, scenario, method, rep_id, horizon, fold_num, output_dir, history_len=300):
    try:
        plt.figure(figsize=(20, 8))
        forecast_start_index = model_results.model.endog.shape[0]
        if forecast_start_index >= len(df_full_imputed): plt.close(); return
        history = df_full_imputed.iloc[:forecast_start_index]
        if original_nan_indices and method != 'Original_Data_Baseline':
            if not any(idx in original_nan_indices for idx in set(history.index)): plt.close(); return
        future_gt = df_full_ground_truth.iloc[forecast_start_index : forecast_start_index + horizon]
        if original_nan_indices:
            if any(idx in original_nan_indices for idx in set(future_gt.index)): plt.close(); return
        if len(future_gt) < horizon: plt.close(); return
        exog_future = df_full_imputed[EXOGENOUS_COLS + TIME_FEATURES].iloc[forecast_start_index : forecast_start_index + horizon]
        forecast_obj = model_results.get_forecast(steps=horizon, exog=exog_future)
        forecast_df = forecast_obj.summary_frame()
        plot_history_display = history.iloc[-history_len:]
        plt.plot(plot_history_display.index, plot_history_display[TARGET_VARIABLE], label='Observed History', color='royalblue')
        plt.plot(future_gt.index, future_gt[TARGET_VARIABLE], label='Ground Truth', color='red')
        plt.plot(forecast_df.index, forecast_df['mean'], label='Forecast', color='green', linestyle='--')
        plt.axvline(x=history.index[-1], color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
        plt.title(f'Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
        plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        scenario_safe = scenario.replace('/', '_').replace('\\', '_')
        plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe)
        os.makedirs(plot_save_dir, exist_ok=True)
        plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe}_{rep_id}_h{horizon}_fold{fold_num}.png'
        plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
        plt.close()
    except Exception:
        plt.close()

# --- FUNCIÓN DE EVALUACIÓN MODIFICADA PARA PLOTEAR TODOS LOS HORIZONTES ---
def train_predict_eval_sarimax_multi_horizon(df_city_data_input, df_city_data_ground_truth, original_nan_indices, target_col, exo_cols, time_features_cols, sarima_order, sarima_seasonal_order, horizons_list, n_cv_splits, plot_params=None):
    results_by_horizon = []
    y = df_city_data_input[target_col]
    X = df_city_data_input[exo_cols + time_features_cols]
    min_len = (sum(sarima_order) + sum(sarima_seasonal_order)) * 2 + max(horizons_list) + n_cv_splits
    if len(y) < min_len:
        for h in horizons_list: results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon
        
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons_list))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons_list}
    
    for fold, (train_index, test_index) in enumerate(tqdm(tscv.split(X), total=n_cv_splits, desc="    CV Folds", leave=False)):
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        if y_train.empty or X_train.empty: continue
        
        results_fit = None # Inicializar
        try:
            model = SARIMAX(y_train, exog=X_train, order=sarima_order, seasonal_order=sarima_seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
            results_fit = model.fit(disp=False, maxiter=200)
            forecasts = results_fit.get_forecast(steps=max(horizons_list), exog=X_test.head(max(horizons_list)))
            forecast_values = forecasts.predicted_mean
            for h in horizons_list:
                y_true_h, y_pred_h = y_test.head(h).values, forecast_values.head(h).values
                if len(y_true_h) < h: continue
                mae = mean_absolute_error(y_true_h, y_pred_h)
                rmse = np.sqrt(mean_squared_error(y_true_h, y_pred_h))
                mase = calculate_mase(y_true_h, y_pred_h, y_train.values)
                accumulated_metrics[h]['maes'].append(mae)
                accumulated_metrics[h]['rmses'].append(rmse)
                if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)
        except Exception:
            continue

        # --- LÓGICA DE PLOTEO MODIFICADA ---
        # Si hay parámetros de ploteo Y es el primer fold Y el modelo se ajustó correctamente
        if plot_params and fold == 0 and results_fit:
            # Iterar sobre cada horizonte para generar un gráfico
            for h_plot in horizons_list:
                # Actualizar el horizonte en los parámetros del plot
                plot_params_horizon = plot_params.copy()
                plot_params_horizon['horizon'] = h_plot
                
                generate_sarimax_forecasting_plots(
                    df_full_imputed=df_city_data_input, 
                    df_full_ground_truth=df_city_data_ground_truth,
                    model_results=results_fit, 
                    original_nan_indices=original_nan_indices, 
                    fold_num=fold + 1, 
                    **plot_params_horizon
                )

    for h in horizons_list:
        results_by_horizon.append({'Horizon': h, 'MAE': np.nanmean(accumulated_metrics[h]['maes']) if accumulated_metrics[h]['maes'] else np.nan, 'RMSE': np.nanmean(accumulated_metrics[h]['rmses']) if accumulated_metrics[h]['rmses'] else np.nan, 'MASE': np.nanmean(accumulated_metrics[h]['mases']) if accumulated_metrics[h]['mases'] else np.nan})
    return results_by_horizon

def process_task(task_params):
    method_name = task_params['method_name']
    city_name = task_params['city_name']
    original_data_path = task_params['original_data_path']
    df_original_full = pd.read_csv(original_data_path)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
        df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce')
    df_original_full['Hour'] = df_original_full['Date'].dt.hour
    df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
    df_original_full['Month'] = df_original_full['Date'].dt.month
    city_name_safe = city_name.replace(" ", "_").replace("-", "_")
    order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe}_{TARGET_VARIABLE}.json')
    try:
        with open(order_file_path, 'r') as f: orders = json.load(f)
        order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
    except Exception:
        order_tuple, seasonal_order_tuple = (1,1,1), (1,1,0,24)

    if method_name == 'Original_Data_Baseline':
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_input.empty: return []
        if df_city_input.isnull().values.any():
            df_city_input.ffill(inplace=True); df_city_input.bfill(inplace=True)
            if df_city_input.isnull().values.any(): return []
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
        horizon_metrics = train_predict_eval_sarimax_multi_horizon(df_city_input, df_city_input, set(), TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_params)
        results = [{'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]
        return results

    scenario_folder = task_params['scenario_folder']
    city_folder = city_name.replace(" ", "_")
    df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
    scenario_path = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs', city_folder, scenario_folder)
    all_files = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))
    if not all_files: return []
    rep_file_path = all_files[0]
    file_name = os.path.basename(rep_file_path)
    rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
    try:
        df_rep = pd.read_csv(rep_file_path)
        df_rep['Date'] = df_rep['Date'].astype(str).str[:19]
        df_rep['Date'] = pd.to_datetime(df_rep['Date'])
        if not all(col in df_rep.columns for col in TIME_FEATURES):
            df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
    except Exception: return []
    if df_rep.isnull().values.any(): df_rep = df_rep.ffill().bfill()
    if df_rep.isnull().values.any(): return []
    plot_params = {'city': city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
    nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id.split('_imputed')[0]}_with_nans.csv" if '_imputed' in rep_file_id else f"{city_folder}_{scenario_folder}_{rep_file_id}_with_nans.csv"
    original_nan_indices = set()
    try:
        df_with_nans = pd.read_csv(os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados', city_folder, scenario_folder, nan_filename))
        original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
    except FileNotFoundError: pass
    horizon_metrics_rep = train_predict_eval_sarimax_multi_horizon(df_rep, df_city_ground_truth, original_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_params)
    results = [{'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics_rep]
    return results

if __name__ == '__main__':
    script_start_time = time.time()
    
    if not os.path.exists(ORIGINAL_DATA_FILE):
        print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'."); exit()
    print(f"Archivo de datos original encontrado en: {ORIGINAL_DATA_FILE}")

    cities_to_process = pd.read_csv(ORIGINAL_DATA_FILE, usecols=['City'])['City'].unique()

    tasks_params = []
    for city_name in cities_to_process:
        tasks_params.append({'method_name': 'Original_Data_Baseline', 'city_name': city_name, 'original_data_path': ORIGINAL_DATA_FILE})

    for method_name in METHODS_TO_PROCESS:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue
        for city_name in cities_to_process:
            city_folder = city_name.replace(" ", "_")
            city_path = os.path.join(imputed_csvs_root, city_folder)
            if not os.path.isdir(city_path): continue
            for scenario_folder in os.listdir(city_path):
                if os.path.isdir(os.path.join(city_path, scenario_folder)):
                    tasks_params.append({
                        'method_name': method_name, 'city_name': city_name, 
                        'scenario_folder': scenario_folder, 'original_data_path': ORIGINAL_DATA_FILE
                    })

    tasks = [delayed(process_task)(params) for params in tasks_params]
    
    print(f"\nSe han generado {len(tasks)} tareas para procesar en paralelo.")
    print("Iniciando procesamiento...")
    
    results_list = Parallel(n_jobs=-1, backend="loky")(
        tqdm(tasks, desc="Progreso General de Tareas")
    )
    
    print("\nProcesamiento en paralelo completado. Recolectando y guardando resultados...")
    all_prediction_metrics_detailed = [item for sublist in results_list for item in sublist if sublist]

    if all_prediction_metrics_detailed:
        print("Creando DataFrame principal y guardando métricas detalladas...")
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        detailed_output_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.to_csv(detailed_output_path, index=False)
        
        print("Calculando y guardando resúmenes...")
        df_summary_by_city = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'), Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'), Count=('MAE', 'count')).reset_index()
        summary_by_city_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
        df_summary_by_city.to_csv(summary_by_city_path, index=False)

        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        global_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
        df_global_summary.to_csv(global_summary_path, index=False)
        
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        avg_over_horizons_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
        df_avg_over_horizons.to_csv(avg_over_horizons_path, index=False)
        
        print("\n--- Resumen Global por Método y Horizonte ---\n", df_global_summary.to_string())
        print("\n--- Resumen Final por Método (promedio de todas las ciudades, escenarios y horizontes) ---\n", df_avg_over_horizons)
    else:
        print("\nNo se generaron resultados para analizar.")

    script_end_time = time.time()
    print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")
    print(f"Gráficos de ejemplo guardados en: {PREDICTION_PLOTS_OUTPUT_DIR}")

Archivo de datos original encontrado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv

Se han generado 222 tareas para procesar en paralelo.
Iniciando procesamiento...


Progreso General de Tareas: 100%|██████████| 222/222 [1:43:53<00:00, 28.08s/it]



Procesamiento en paralelo completado. Recolectando y guardando resultados...
Creando DataFrame principal y guardando métricas detalladas...
Calculando y guardando resúmenes...

--- Resumen Global por Método y Horizonte ---
          Imputation_Method  Horizon_h  Global_Avg_MAE  Global_Avg_RMSE  Global_Avg_MASE
0                 LSTM-VAE         12        4.981983         5.677763         3.325369
1                 LSTM-VAE         24        7.655001         8.752720         5.172912
2                 LSTM-VAE         36        9.038143        10.191256         6.180986
3                 LSTM-VAE         48        9.576597        10.752805         6.604639
4                 LSTM-VAE         60        9.688014        10.919364         6.735352
5     Linear Interpolation         12        4.445621         5.045797         3.711385
6     Linear Interpolation         24        7.689384         8.855142         6.589463
7     Linear Interpolation         36        9.340445        10.516692 

Random Forest

In [2]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de Estilo para Gráficas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['lines.linewidth'] = 2.5

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]

# --- PARÁMETROS DE EJECUCIÓN ---
N_LAGS_AQI = 120
N_LAGS_EXO = 120
N_PLOT_EXAMPLES = 1
HISTORY_LEN_FOR_PLOT = 300
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Finetuned',
    'MOMENT Pretrained', 'Random Forest', 'SARIMA'
]

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'RandomForest_full_FT'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')

os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_OUTPUT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def create_multivariate_lagged_features_rf(df, target_col, exo_cols, time_cols, n_lags_target, n_lags_exo, horizon):
    features = []
    for i in range(horizon, n_lags_target + horizon):
        features.append(df[target_col].shift(i).rename(f'{target_col}_lag{i}'))
    for exo_col in exo_cols:
        if exo_col != target_col:
            for i in range(horizon, n_lags_exo + horizon):
                features.append(df[exo_col].shift(i).rename(f'{exo_col}_lag{i}'))
    for tf_col in time_cols:
        if tf_col in df.columns:
            features.append(df[tf_col])
    return pd.concat(features, axis=1)

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

# --- NUEVA FUNCIÓN PARA ENCONTRAR BLOQUES DE NANS ---
def find_nan_blocks(original_nan_indices):
    if not original_nan_indices:
        return []
    nan_list = sorted(list(original_nan_indices))
    if not nan_list:
        return []
    blocks = []
    start_block = nan_list[0]
    for i in range(1, len(nan_list)):
        if nan_list[i] != nan_list[i-1] + 1:
            blocks.append((start_block, nan_list[i-1]))
            start_block = nan_list[i]
    blocks.append((start_block, nan_list[-1]))
    return blocks

def generate_rf_forecasting_plots(df_imputed_data, df_ground_truth_data, model, X_all_features_with_nans, nan_blocks, city, scenario, method, rep_id, horizon, output_dir, n_examples=1, history_len=300):
    max_lags_h = max(N_LAGS_AQI, N_LAGS_EXO) + horizon
    possible_points = range(max_lags_h + history_len, len(df_imputed_data) - horizon - 1)
    
    valid_points = []
    if not nan_blocks or method == 'Original_Data_Baseline':
        valid_points = list(possible_points)
    else:
        for p in possible_points:
            history_start, history_end = p - max_lags_h, p - horizon
            future_indices = range(p, p + horizon)
            # Condición 1: Ningún NaN en el futuro
            if not any(block_start <= i <= block_end for i in future_indices for block_start, block_end in nan_blocks):
                 # Condición 2: Al menos un bloque de NaNs COMPLETAMENTE contenido en el historial
                if any(block_start >= history_start and block_end <= history_end for block_start, block_end in nan_blocks):
                    valid_points.append(p)

    if not valid_points: return
    example_indices = random.sample(valid_points, min(n_examples, len(valid_points)))
    
    for split_point in example_indices:
        try:
            plt.figure(figsize=(20, 8))
            future_gt_df = df_ground_truth_data.iloc[split_point:split_point + horizon]
            X_forecast_clean = X_all_features_with_nans.loc[future_gt_df.index].dropna()
            if X_forecast_clean.empty: plt.close(); continue
            
            predictions = model.predict(X_forecast_clean)
            preds_series = pd.Series(predictions, index=X_forecast_clean.index)
            
            plot_history = df_imputed_data.iloc[split_point-history_len:split_point]
            plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
            plt.plot(future_gt_df.index, future_gt_df[TARGET_VARIABLE], label='Ground Truth', color='red')
            plt.plot(preds_series.index, preds_series, label='Forecast', color='green', linestyle='--')
            
            plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
            plt.title(f'Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
            plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
            plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
            
            scenario_safe = scenario.replace('/', '_').replace('\\', '_')
            plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe)
            os.makedirs(plot_save_dir, exist_ok=True)
            plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe}_{rep_id}_h{horizon}.png'
            plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
            plt.close()
        except Exception:
            plt.close()

def train_predict_eval_rf_multi_horizon(df_imputed, df_ground_truth, nan_blocks, target_col, exo_cols, time_features, n_lags_target, n_lags_exo, horizons, plot_params=None):
    results_by_horizon = []
    
    for horizon in tqdm(horizons, desc="    Horizons", leave=False):
        X_with_nans = create_multivariate_lagged_features_rf(df_imputed, target_col, exo_cols, time_features, n_lags_target, n_lags_exo, horizon)
        y = df_ground_truth[target_col]

        train_data = pd.concat([X_with_nans, y], axis=1).dropna()
        X_train_clean, y_train_clean = train_data.drop(columns=[target_col]), train_data[target_col]

        if X_train_clean.empty or len(X_train_clean) < 200:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        model = RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_split=5, min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=1)
        model.fit(X_train_clean, y_train_clean)
        
        if plot_params:
            plot_params_h = plot_params.copy()
            plot_params_h['horizon'] = horizon
            generate_rf_forecasting_plots(df_imputed, df_ground_truth, model, X_with_nans, nan_blocks, **plot_params_h)
            
        max_lags_h = max(n_lags_target, n_lags_exo) + horizon
        predictable_indices = X_with_nans.dropna().index
        
        impact_indices = []
        if not nan_blocks or plot_params.get('method') == 'Original_Data_Baseline':
            impact_indices = predictable_indices.tolist()
        else:
            for idx in predictable_indices:
                history_start, history_end = idx - max_lags_h, idx - horizon
                if any(block_start >= history_start and block_end <= history_end for block_start, block_end in nan_blocks):
                    impact_indices.append(idx)
        
        if not impact_indices:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue
            
        X_eval = X_with_nans.loc[impact_indices]
        y_eval_true = y.loc[impact_indices]
        predictions = model.predict(X_eval)
        
        mae = mean_absolute_error(y_eval_true, predictions)
        rmse = np.sqrt(mean_squared_error(y_eval_true, predictions))
        mase = calculate_mase(y_eval_true.values, predictions, y_train_clean)
        
        results_by_horizon.append({'Horizon': horizon, 'MAE': mae, 'RMSE': rmse, 'MASE': mase})
        
    return results_by_horizon

def process_rf_task(task_params):
    method_name = task_params['method_name']
    city_name = task_params['city_name']
    original_data_path = task_params['original_data_path']
    df_original_full = pd.read_csv(original_data_path)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
        df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce').ffill().bfill()
    df_original_full['Hour'], df_original_full['DayOfWeek'], df_original_full['Month'] = df_original_full['Date'].dt.hour, df_original_full['Date'].dt.dayofweek, df_original_full['Date'].dt.month

    if method_name == 'Original_Data_Baseline':
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_input.isnull().values.any(): return []
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
        horizon_metrics = train_predict_eval_rf_multi_horizon(df_city_input, df_city_input, [], TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, plot_params=plot_params)
        return [{'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

    scenario_folder = task_params['scenario_folder']
    city_folder = city_name.replace(" ", "_")
    df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
    scenario_path = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs', city_folder, scenario_folder)
    try:
        rep_file_path = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[0]
    except IndexError: return []

    try:
        df_rep = pd.read_csv(rep_file_path)
        df_rep['Date'] = df_rep['Date'].astype(str).str[:19]
        df_rep['Date'] = pd.to_datetime(df_rep['Date'])
        if not all(col in df_rep.columns for col in TIME_FEATURES):
            df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
    except Exception: return []
    
    rep_file_id = os.path.basename(rep_file_path)
    plot_params = {'city': city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
    
    nan_blocks = []
    try:
        nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id.split('_imputed')[0]}_with_nans.csv" if '_imputed' in rep_file_id else f"{city_folder}_{scenario_folder}_{rep_file_id}_with_nans.csv"
        df_with_nans = pd.read_csv(os.path.join(SIMULATED_DATA_ROOT_DIR, city_folder, scenario_folder, nan_filename))
        original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
        nan_blocks = find_nan_blocks(original_nan_indices)
    except FileNotFoundError: pass

    horizon_metrics = train_predict_eval_rf_multi_horizon(df_rep, df_city_ground_truth, nan_blocks, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, plot_params=plot_params)
    return [{'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

if __name__ == '__main__':
    script_start_time = time.time()
    if not os.path.exists(ORIGINAL_DATA_FILE): exit()
    print(f"Archivo de datos original encontrado: {ORIGINAL_DATA_FILE}")

    cities_to_process = pd.read_csv(ORIGINAL_DATA_FILE, usecols=['City'])['City'].unique()
    tasks = []
    for city_name in cities_to_process:
        tasks.append(delayed(process_rf_task)({'method_name': 'Original_Data_Baseline', 'city_name': city_name, 'original_data_path': ORIGINAL_DATA_FILE}))

    for method_name in METHODS_TO_PROCESS:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue
        for city_name in cities_to_process:
            city_folder = city_name.replace(" ", "_")
            city_path = os.path.join(imputed_csvs_root, city_folder)
            if not os.path.isdir(city_path): continue
            for scenario_folder in os.listdir(city_path):
                if os.path.isdir(os.path.join(city_path, scenario_folder)):
                    tasks.append(delayed(process_rf_task)({'method_name': method_name, 'city_name': city_name, 'scenario_folder': scenario_folder, 'original_data_path': ORIGINAL_DATA_FILE}))

    print(f"\nSe han generado {len(tasks)} tareas para procesar en paralelo.")
    results_list = Parallel(n_jobs=-1, backend="loky")(tqdm(tasks, desc="Procesando Tareas RF"))
    
    print("\nProcesamiento completado. Generando resúmenes...")
    all_prediction_metrics_detailed = [item for sublist in results_list for item in sublist if sublist]

    if all_prediction_metrics_detailed:
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        df_all_metrics.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("\n[OK] Métricas detalladas guardadas.")
        
        df_summary_by_city = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index()
        df_summary_by_city.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen por ciudad guardado.")

        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        df_global_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen global guardado.")
        
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        df_avg_over_horizons.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen final guardado.")
        
        print("\n--- Resumen Final (promedio de todo) ---\n", df_avg_over_horizons)
    else:
        print("\nNo se generaron resultados.")

    print(f"\nScript completado. Tiempo total: {(time.time() - script_start_time)/60:.2f} minutos.")

Archivo de datos original encontrado: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv

Se han generado 222 tareas para procesar en paralelo.


Procesando Tareas RF: 100%|██████████| 222/222 [20:54<00:00,  5.65s/it]



Procesamiento completado. Generando resúmenes...

[OK] Métricas detalladas guardadas.
[OK] Resumen por ciudad guardado.
[OK] Resumen global guardado.
[OK] Resumen final guardado.

--- Resumen Final (promedio de todo) ---
       Imputation_Method  Final_Avg_MAE  Final_Avg_RMSE  Final_Avg_MASE
0              LSTM-VAE       2.144511        2.813504        1.757184
1  Linear Interpolation       1.873474        2.477973        1.531718
2      MOMENT Finetuned       2.139072        2.819296        1.759486
3     MOMENT Pretrained       2.185641        2.878981        1.798909
4         Random Forest       2.133888        2.806147        1.744218
5                SARIMA       2.004710        2.641718        1.640359

Script completado. Tiempo total: 25.81 minutos.


RF baseline

In [7]:
# =============================================================================
# CELDA AUTOCONTENIDA Y CORREGIDA PARA EJECUTAR EL BASELINE DE RANDOM FOREST
# SOLUCIÓN: Eliminar la columna 'CO2' vacía inmediatamente después de la carga.
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (20, 8), 'axes.titlesize': 22, 'axes.labelsize': 18,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 16,
    'lines.linewidth': 2.5
})

# --- PARÁMETROS ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_LAGS_AQI = 120
N_LAGS_EXO = 120
HISTORY_LEN_FOR_PLOT = 300
PREDICTION_METHOD_NAME = 'RandomForest_full_FT'

# --- RUTAS ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')
os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_OUTPUT_DIR, exist_ok=True)


# --- FUNCIONES AUXILIARES ---
def create_multivariate_lagged_features_rf(df, target_col, exo_cols, time_cols, n_lags_target, n_lags_exo, horizon):
    features = []
    for i in range(horizon, n_lags_target + horizon):
        features.append(df[target_col].shift(i).rename(f'{target_col}_lag{i}'))
    for exo_col in exo_cols:
        if exo_col != target_col:
            for i in range(horizon, n_lags_exo + horizon):
                features.append(df[exo_col].shift(i).rename(f'{exo_col}_lag{i}'))
    for tf_col in time_cols:
        if tf_col in df.columns:
            features.append(df[tf_col])
    return pd.concat(features, axis=1)

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

def generate_rf_forecasting_plots(df_imputed_data, df_ground_truth_data, model, X_all_features_with_nans, city, scenario, method, rep_id, horizon, output_dir, history_len=300):
    max_lags_h = max(N_LAGS_AQI, N_LAGS_EXO) + horizon
    possible_points = range(max_lags_h + history_len, len(df_imputed_data) - horizon - 1)
    if not list(possible_points): return
    
    split_point = random.choice(list(possible_points))
    try:
        plt.figure(figsize=(20, 8))
        future_gt_df = df_ground_truth_data.iloc[split_point:split_point + horizon]
        X_forecast_clean = X_all_features_with_nans.loc[future_gt_df.index].dropna()
        if X_forecast_clean.empty: plt.close(); return
        
        predictions = model.predict(X_forecast_clean)
        preds_series = pd.Series(predictions, index=X_forecast_clean.index)
        
        plot_history = df_imputed_data.iloc[split_point-history_len:split_point]
        plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
        plt.plot(future_gt_df.index, future_gt_df[TARGET_VARIABLE], label='Ground Truth', color='red')
        plt.plot(preds_series.index, preds_series, label='Forecast', color='green', linestyle='--')
        
        plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
        plt.title(f'Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
        plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        
        scenario_safe = scenario.replace('/', '_').replace('\\', '_')
        plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe)
        os.makedirs(plot_save_dir, exist_ok=True)
        plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe}_{rep_id}_h{horizon}.png'
        plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
        plt.close()
    except Exception as e:
        print(f"      [ERROR en PLOT] {e}")
        plt.close()

def train_predict_eval_rf_multi_horizon(df_imputed, df_ground_truth, target_col, exo_cols, time_features, n_lags_target, n_lags_exo, horizons, plot_params=None):
    results_by_horizon = []
    
    for horizon in tqdm(horizons, desc=f"    Horizons for {plot_params['city']}", leave=False):
        X_with_nans = create_multivariate_lagged_features_rf(df_imputed, target_col, exo_cols, time_features, n_lags_target, n_lags_exo, horizon)
        y = df_ground_truth[target_col]

        train_data = pd.concat([X_with_nans, y], axis=1).dropna()
        X_train_clean, y_train_clean = train_data.drop(columns=[target_col]), train_data[target_col]
        
        if X_train_clean.empty or len(X_train_clean) < 200:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        model = RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_split=5, min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=1)
        model.fit(X_train_clean, y_train_clean)
        
        if plot_params:
            generate_rf_forecasting_plots(df_imputed, df_ground_truth, model, X_with_nans, **plot_params, horizon=horizon)
            
        predictable_indices = X_with_nans.dropna().index
        if predictable_indices.empty:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue
            
        X_eval, y_eval_true = X_with_nans.loc[predictable_indices], y.loc[predictable_indices]
        predictions = model.predict(X_eval)
        
        results_by_horizon.append({
            'Horizon': horizon,
            'MAE': mean_absolute_error(y_eval_true, predictions),
            'RMSE': np.sqrt(mean_squared_error(y_eval_true, predictions)),
            'MASE': calculate_mase(y_eval_true.values, predictions, y_train_clean)
        })
    return results_by_horizon

# --- SCRIPT PRINCIPAL SECUENCIAL ---
if __name__ == '__main__':
    print("--- INICIANDO EJECUCIÓN DEL BASELINE PARA RANDOM FOREST (SECUENCIAL) ---")
    script_start_time_baseline = time.time()
    
    try:
        df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
        
        # --- SOLUCIÓN: Eliminar la columna 'CO2' vacía inmediatamente después de la carga ---
        if 'CO2' in df_original_full.columns:
            df_original_full = df_original_full.drop(columns=['CO2'])
            print("Columna 'CO2' eliminada.")

        df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
        for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
            if col in df_original_full.columns:
                df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce')
                df_original_full[col] = df_original_full[col].interpolate(method='linear', limit_direction='both').ffill().bfill()
        
        df_original_full['Hour'] = df_original_full['Date'].dt.hour
        df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
        df_original_full['Month'] = df_original_full['Date'].dt.month
        print("Datos originales cargados y preparados.")
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo original en {ORIGINAL_DATA_FILE}. Saliendo."); exit()

    cities_to_process = df_original_full['City'].unique()
    baseline_metrics_list = []
    
    print(f"\nProcesando {len(cities_to_process)} ciudades para el escenario 'Original_Data_Baseline'...")
    for city_name in cities_to_process:
        print(f"\n--- Procesando Ciudad: {city_name} ---")
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        
        if df_city_input.empty or df_city_input.isnull().values.any():
            print(f"  -> ¡ERROR! Datos para {city_name} vacíos o con NaNs. Omitiendo.")
            continue
            
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
        
        horizon_metrics = train_predict_eval_rf_multi_horizon(
            df_imputed=df_city_input, df_ground_truth=df_city_input, target_col=TARGET_VARIABLE, 
            exo_cols=EXOGENOUS_COLS, time_features=TIME_FEATURES, n_lags_target=N_LAGS_AQI, 
            n_lags_exo=N_LAGS_EXO, horizons=PREDICTION_HORIZONS, plot_params=plot_params
        )
        
        for hm in horizon_metrics:
            baseline_metrics_list.append({'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], **hm})

    print("\nProcesamiento del Baseline completado. Actualizando archivos de métricas...")

    if baseline_metrics_list:
        df_baseline_metrics = pd.DataFrame(baseline_metrics_list)
        detailed_results_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')

        try:
            df_existing_metrics = pd.read_csv(detailed_results_path)
            df_existing_metrics = df_existing_metrics[df_existing_metrics['Imputation_Method'] != 'Original_Data_Baseline']
            df_all_metrics = pd.concat([df_existing_metrics, df_baseline_metrics], ignore_index=True)
            print("Archivo de métricas existente encontrado y actualizado.")
        except FileNotFoundError:
            print("No se encontró archivo de métricas existente. Creando uno nuevo.")
            df_all_metrics = df_baseline_metrics
        
        df_all_metrics.to_csv(detailed_results_path, index=False)
        print(f"\n[OK] Archivo de métricas detalladas actualizado: {detailed_results_path}")

        # Regenerar todos los archivos de resumen...
        df_summary = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index()
        df_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen por ciudad regenerado.")
        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        df_global_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen global regenerado.")
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        df_avg_over_horizons.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen final regenerado.")
        
        print("\n--- Resumen Final (Actualizado) ---\n", df_avg_over_horizons.sort_values(by='Final_Avg_MAE'))
    else:
        print("\nNo se generaron resultados para el Baseline.")

    print(f"\nScript de Baseline completado. Tiempo total: {(time.time() - script_start_time_baseline)/60:.2f} minutos.")

--- INICIANDO EJECUCIÓN DEL BASELINE PARA RANDOM FOREST (SECUENCIAL) ---
Columna 'CO2' eliminada.
Datos originales cargados y preparados.

Procesando 6 ciudades para el escenario 'Original_Data_Baseline'...

--- Procesando Ciudad: Brasilia ---



--- Procesando Ciudad: Cairo ---



--- Procesando Ciudad: Dubai ---



--- Procesando Ciudad: London ---



--- Procesando Ciudad: New York ---



--- Procesando Ciudad: Sydney ---



Procesamiento del Baseline completado. Actualizando archivos de métricas...
Archivo de métricas existente encontrado y actualizado.

[OK] Archivo de métricas detalladas actualizado: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\predicciones\RandomForest_full_FT\metricas\ALL_prediction_metrics_detailed_RandomForest_full_FT.csv
[OK] Resumen por ciudad regenerado.
[OK] Resumen global regenerado.
[OK] Resumen final regenerado.

--- Resumen Final (Actualizado) ---
         Imputation_Method  Final_Avg_MAE  Final_Avg_RMSE  Final_Avg_MASE
1    Linear Interpolation       1.873474        2.477973        1.531718
6                  SARIMA       2.004710        2.641718        1.640359
4  Original_Data_Baseline       2.080170        2.734254        1.706095
5           Random Forest       2.133888        2.806147        1.744218
2        MOMENT Finetuned       2.139072        2.819296        1.759486
0                LSTM-VAE       2.144511        2.813504        1.757184
3      

LSTM

In [10]:
import pandas as pd
import numpy as np
import os
import random
import tensorflow as tf
from keras import layers, Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- CONFIGURACIÓN DE GPU ---
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

# --- PARÁMETROS DE CONFIGURACIÓN Y MODELO ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_LAGS = 120
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Finetuned',
    'MOMENT Pretrained', 'Random Forest', 'SARIMA'
]

# Parámetros de ejecución y ploteo
N_PLOT_EXAMPLES = 1
HISTORY_LEN_FOR_PLOT = 300
N_WFV_FOLDS = 3

# Hiperparámetros del modelo LSTM
LSTM_UNITS = 64
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001
CLIPNORM = 1.0
EPOCHS = 40
BATCH_SIZE = 128

# --- NUEVO: Hiperparámetros para Callbacks ---
ES_PATIENCE = 12  # Paciencia para Early Stopping
LR_PATIENCE = 5   # Paciencia para reducir el Learning Rate


# --- RUTAS ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'LSTM_full_FT'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')
MODELS_TEMP_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'temp_models')
for d in [PREDICTION_METRICS_OUTPUT_DIR, PREDICTION_PLOTS_OUTPUT_DIR, MODELS_TEMP_DIR]: os.makedirs(d, exist_ok=True)

# --- FUNCIONES AUXILIARES ---
def create_sequences_single_shot(input_data, target_data, n_lags, horizon):
    X, y, indices = [], [], []
    y_multistep = np.lib.stride_tricks.as_strided(
        target_data,
        shape=(len(target_data) - horizon + 1, horizon),
        strides=(target_data.strides[0],) * 2
    )
    end_idx = len(input_data) - n_lags - horizon + 1
    for i in range(end_idx):
        X.append(input_data[i:(i + n_lags)])
    X = np.array(X)
    y = y_multistep[n_lags:n_lags + len(X)]
    indices = np.arange(n_lags, n_lags + len(X))
    return X, y, indices

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

def generate_lstm_forecasting_plots_single_shot(df_imputed_data, df_ground_truth_data, model, scaler_all, scaler_target, city, scenario, method, rep_id, output_dir, history_len=300):
    try:
        max_horizon = max(PREDICTION_HORIZONS)
        split_point = len(df_imputed_data) - history_len - max_horizon
        if split_point < N_LAGS: return
        all_cols = EXOGENOUS_COLS + TIME_FEATURES + [TARGET_VARIABLE]
        
        history_df = df_imputed_data.iloc[split_point - N_LAGS : split_point]
        input_seq = scaler_all.transform(history_df[all_cols])
        input_seq = input_seq.reshape(1, N_LAGS, input_seq.shape[1])
        prediction_scaled = model.predict(input_seq, verbose=0)
        prediction_unscaled = scaler_target.inverse_transform(prediction_scaled).flatten()
        
        for horizon in PREDICTION_HORIZONS:
            plt.figure(figsize=(20, 8))
            future_gt_df = df_ground_truth_data.iloc[split_point : split_point + horizon]
            future_pred = prediction_unscaled[:horizon]
            plot_history = df_imputed_data.iloc[split_point-history_len:split_point]
            plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
            plt.plot(future_gt_df.index, future_gt_df[TARGET_VARIABLE], label='Ground Truth', color='red')
            plt.plot(future_gt_df.index, future_pred, label='Forecast', color='green', linestyle='--')
            plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
            plt.title(f'Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
            plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
            plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
            scenario_safe = scenario.replace('/', '_').replace('\\', '_')
            plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe)
            os.makedirs(plot_save_dir, exist_ok=True)
            plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe}_{rep_id}_h{horizon}.png'
            plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
            plt.close()
    except Exception:
        plt.close()

def train_predict_eval_lstm_wfv_single_shot(df_imputed, df_ground_truth, original_nan_indices, target_col, all_features_cols, n_lags, horizons, plot_params=None):
    output_horizon = max(horizons)
    scaler_all = MinMaxScaler(); scaled_data = scaler_all.fit_transform(df_imputed[all_features_cols])
    scaler_target = MinMaxScaler(); scaled_target = scaler_target.fit_transform(df_imputed[[target_col]])
    
    train_val_size = int(len(scaled_data) * 0.8)
    train_val_data = scaled_data[:train_val_size]
    test_data = scaled_data[train_val_size:]
    
    tf.keras.backend.clear_session()
    X_tv, y_tv, _ = create_sequences_single_shot(train_val_data, scaled_target.flatten(), n_lags, output_horizon)
    
    if len(X_tv) < 200: return []
    
    tscv = TimeSeriesSplit(n_splits=N_WFV_FOLDS)
    best_val_loss = float('inf')
    model_path = os.path.join(MODELS_TEMP_DIR, f"best_model_{plot_params['city']}_{plot_params['method']}_{plot_params['scenario']}.keras")

    for fold, (train_idx, val_idx) in enumerate(tqdm(tscv.split(X_tv), total=N_WFV_FOLDS, desc="    WFV Folds", leave=False)):
        X_train, y_train = X_tv[train_idx], y_tv[train_idx]
        X_val, y_val = X_tv[val_idx], y_tv[val_idx]
        
        inputs = layers.Input(shape=(X_train.shape[1], X_train.shape[2]))
        lstm_1 = layers.LSTM(LSTM_UNITS, return_sequences=True)(inputs); lstm_1 = layers.Dropout(DROPOUT_RATE)(lstm_1)
        lstm_2 = layers.LSTM(LSTM_UNITS // 2)(lstm_1); lstm_2 = layers.Dropout(DROPOUT_RATE)(lstm_2)
        outputs = layers.Dense(output_horizon)(lstm_2)
        model = Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=CLIPNORM)
        model.compile(optimizer=optimizer, loss='mae')
        
        # --- NUEVOS CALLBACKS PARA MONITORIZACIÓN ---
        es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=ES_PATIENCE, restore_best_weights=True, verbose=0)
        lr_scheduler_callback = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=LR_PATIENCE, factor=0.5, min_lr=1e-6, verbose=0)
        print_callback = tf.keras.callbacks.LambdaCallback(on_epoch_end=lambda epoch, logs: print(f" - Epoch {epoch+1:02d}: val_loss: {logs['val_loss']:.4f}", end=""))
        
        print(f"\n      Fold {fold+1}: Entrenando...")
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[es_callback, lr_scheduler_callback, print_callback], verbose=0)
        
        val_loss = min(history.history['val_loss'])
        print(f"\n      Fold {fold+1} completado. Mejor val_loss: {val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print("      -> Nuevo mejor modelo encontrado. Guardando...")
            model.save(model_path)
    
    if not os.path.exists(model_path): return []
    best_model = tf.keras.models.load_model(model_path)
    if plot_params:
        generate_lstm_forecasting_plots_single_shot(df_imputed, df_ground_truth, best_model, scaler_all, scaler_target, **plot_params)
        
    X_test, y_test, indices_test = create_sequences_single_shot(test_data, scaled_target.flatten(), n_lags, output_horizon)
    if len(X_test) == 0: return []
    
    impact_indices = []
    if not original_nan_indices or plot_params.get('method') == 'Original_Data_Baseline':
        impact_indices = list(range(len(X_test)))
    else:
        for i, original_idx in enumerate(indices_test):
            history_window = range(original_idx + train_val_size - n_lags, original_idx + train_val_size)
            if any(h_idx in original_nan_indices for h_idx in history_window):
                impact_indices.append(i)
    
    if not impact_indices: return []

    X_eval = X_test[impact_indices]
    y_eval_true_scaled = y_test[impact_indices]
    predictions_scaled = best_model.predict(X_eval, verbose=0)
    
    results_by_horizon = []
    y_train_unscaled = scaler_target.inverse_transform(y_tv[train_idx])
    for h in horizons:
        pred_h_unscaled = scaler_target.inverse_transform(predictions_scaled[:, :h])
        true_h_unscaled = scaler_target.inverse_transform(y_eval_true_scaled[:, :h])
        mae = np.mean(np.abs(true_h_unscaled - pred_h_unscaled))
        rmse = np.sqrt(np.mean((true_h_unscaled - pred_h_unscaled)**2))
        mase = calculate_mase(true_h_unscaled.flatten(), pred_h_unscaled.flatten(), y_train_unscaled.flatten())
        results_by_horizon.append({'Horizon': h, 'MAE': mae, 'RMSE': rmse, 'MASE': mase})
        
    return results_by_horizon

def process_lstm_task(task_params):
    method_name, city_name, original_data_path = task_params['method_name'], task_params['city_name'], task_params['original_data_path']
    df_original_full = pd.read_csv(original_data_path)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
        df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce').ffill().bfill()
    df_original_full['Hour'], df_original_full['DayOfWeek'], df_original_full['Month'] = df_original_full['Date'].dt.hour, df_original_full['Date'].dt.dayofweek, df_original_full['Date'].dt.month

    if method_name == 'Original_Data_Baseline':
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_input.isnull().values.any(): return []
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
        horizon_metrics = train_predict_eval_lstm_wfv_single_shot(df_city_input, df_city_input, set(), TARGET_VARIABLE, EXOGENOUS_COLS + TIME_FEATURES + [TARGET_VARIABLE], N_LAGS, PREDICTION_HORIZONS, plot_params=plot_params)
        return [{'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

    scenario_folder = task_params['scenario_folder']
    city_folder = city_name.replace(" ", "_")
    df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
    scenario_path = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs', city_folder, scenario_folder)
    try: rep_file_path = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[0]
    except IndexError: return []
    try:
        df_rep = pd.read_csv(rep_file_path)
        df_rep['Date'] = df_rep['Date'].astype(str).str[:19]
        df_rep['Date'] = pd.to_datetime(df_rep['Date'])
        if not all(col in df_rep.columns for col in TIME_FEATURES):
            df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
        if df_rep.isnull().values.any(): df_rep = df_rep.ffill().bfill()
        if df_rep.isnull().values.any(): return []
    except Exception: return []
    
    rep_file_id = os.path.basename(rep_file_path)
    plot_params = {'city': city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR}
    nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id.split('_imputed')[0]}_with_nans.csv" if '_imputed' in rep_file_id else f"{city_folder}_{scenario_folder}_{rep_file_id}_with_nans.csv"
    original_nan_indices = set()
    try:
        df_with_nans = pd.read_csv(os.path.join(SIMULATED_DATA_ROOT_DIR, city_folder, scenario_folder, nan_filename))
        original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
    except FileNotFoundError: pass

    horizon_metrics = train_predict_eval_lstm_wfv_single_shot(df_rep, df_city_ground_truth, original_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS + TIME_FEATURES + [TARGET_VARIABLE], N_LAGS, PREDICTION_HORIZONS, plot_params=plot_params)
    return [{'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']} for hm in horizon_metrics]

if __name__ == '__main__':
    script_start_time = time.time()
    if not os.path.exists(ORIGINAL_DATA_FILE): exit()
    print(f"Archivo de datos original encontrado: {ORIGINAL_DATA_FILE}")

    cities_to_process = pd.read_csv(ORIGINAL_DATA_FILE, usecols=['City'])['City'].unique()
    tasks_params = []
    for city_name in cities_to_process:
        tasks_params.append({'method_name': 'Original_Data_Baseline', 'city_name': city_name, 'original_data_path': ORIGINAL_DATA_FILE})

    for method_name in METHODS_TO_PROCESS:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue
        for city_name in cities_to_process:
            city_folder = city_name.replace(" ", "_")
            city_path = os.path.join(imputed_csvs_root, city_folder)
            if not os.path.isdir(city_path): continue
            for scenario_folder in os.listdir(city_path):
                if os.path.isdir(os.path.join(city_path, scenario_folder)):
                    tasks_params.append({'method_name': method_name, 'city_name': city_name, 'scenario_folder': scenario_folder, 'original_data_path': ORIGINAL_DATA_FILE})

    tasks = [delayed(process_lstm_task)(params) for params in tasks_params]
    
    print(f"\nSe han generado {len(tasks)} tareas para procesar en paralelo.")
    results_list = Parallel(n_jobs=1)(tqdm(tasks, desc="Procesando Tareas LSTM"))
    
    print("\nProcesamiento completado. Generando resúmenes...")
    all_prediction_metrics_detailed = [item for sublist in results_list for item in sublist if sublist]

    if all_prediction_metrics_detailed:
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        df_all_metrics.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("\n[OK] Métricas detalladas guardadas.")
        
        df_summary = df_all_metrics.groupby(['Imputation_Method', 'City', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index()
        df_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_and_scenario_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen por ciudad y escenario guardado.")
        
        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        df_global_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen global guardado.")
        
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        df_avg_over_horizons.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen final guardado.")
        
        print("\n--- Resumen Final (promedio de todo) ---\n", df_avg_over_horizons)
    else:
        print("\nNo se generaron resultados.")

    print(f"\nScript completado. Tiempo total: {(time.time() - script_start_time)/60:.2f} minutos.")

Archivo de datos original encontrado: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv

Se han generado 222 tareas para procesar en paralelo.


Procesando Tareas LSTM:   3%|▎         | 6/222 [00:00<00:25,  8.49it/s]


      Fold 1: Entrenando...
 - Epoch 01: val_loss: 0.1383 - Epoch 02: val_loss: 0.1051 - Epoch 03: val_loss: 0.0933 - Epoch 04: val_loss: 0.0884 - Epoch 05: val_loss: 0.0892 - Epoch 06: val_loss: 0.0871 - Epoch 07: val_loss: 0.0843 - Epoch 08: val_loss: 0.0817 - Epoch 09: val_loss: 0.0733 - Epoch 10: val_loss: 0.0707 - Epoch 11: val_loss: 0.0703 - Epoch 12: val_loss: 0.0722 - Epoch 13: val_loss: 0.0676 - Epoch 14: val_loss: 0.0707 - Epoch 15: val_loss: 0.0746 - Epoch 16: val_loss: 0.0728 - Epoch 17: val_loss: 0.0775 - Epoch 18: val_loss: 0.0808 - Epoch 19: val_loss: 0.0804 - Epoch 20: val_loss: 0.0889 - Epoch 21: val_loss: 0.0931 - Epoch 22: val_loss: 0.0939 - Epoch 23: val_loss: 0.0855 - Epoch 24: val_loss: 0.0940

 - Epoch 25: val_loss: 0.0871
      Fold 1 completado. Mejor val_loss: 0.0676
      -> Nuevo mejor modelo encontrado. Guardando...

      Fold 2: Entrenando...
 - Epoch 01: val_loss: 0.1221 - Epoch 02: val_loss: 0.1022 - Epoch 03: val_loss: 0.0948 - Epoch 04: val_loss: 0.0875 - Epoch 05: val_loss: 0.0804 - Epoch 06: val_loss: 0.0742 - Epoch 07: val_loss: 0.0791 - Epoch 08: val_loss: 0.0744 - Epoch 09: val_loss: 0.0776 - Epoch 10: val_loss: 0.0772 - Epoch 11: val_loss: 0.0615 - Epoch 12: val_loss: 0.0803 - Epoch 13: val_loss: 0.0741 - Epoch 14: val_loss: 0.0660 - Epoch 15: val_loss: 0.0700 - Epoch 16: val_loss: 0.0642 - Epoch 17: val_loss: 0.0661 - Epoch 18: val_loss: 0.0649 - Epoch 19: val_loss: 0.0617 - Epoch 20: val_loss: 0.0627 - Epoch 21: val_loss: 0.0582 - Epoch 22: val_loss: 0.0536 - Epoch 23: val_loss: 0.0557 - Epoch 24: val_loss: 0.0653 - Epoch 25: val_loss: 0.0527 - Epoch 26: val_loss: 0.0538 - Epoch 27: val_loss: 0.0576 - Epoch 28: val_loss: 0.0547 - Epoch 29: val_loss: 0.054

 - Epoch 37: val_loss: 0.0533
      Fold 2 completado. Mejor val_loss: 0.0527
      -> Nuevo mejor modelo encontrado. Guardando...

      Fold 3: Entrenando...
 - Epoch 01: val_loss: 0.2249 - Epoch 02: val_loss: 0.1790 - Epoch 03: val_loss: 0.1580 - Epoch 04: val_loss: 0.1529 - Epoch 05: val_loss: 0.1285 - Epoch 06: val_loss: 0.1156 - Epoch 07: val_loss: 0.1557 - Epoch 08: val_loss: 0.1310 - Epoch 09: val_loss: 0.1663 - Epoch 10: val_loss: 0.1415 - Epoch 11: val_loss: 0.1491 - Epoch 12: val_loss: 0.1609 - Epoch 13: val_loss: 0.1695 - Epoch 14: val_loss: 0.1656 - Epoch 15: val_loss: 0.1710 - Epoch 16: val_loss: 0.1684 - Epoch 17: val_loss: 0.1707

...

 - Epoch 23: val_loss: 0.0830
      Fold 1 completado. Mejor val_loss: 0.0809
      -> Nuevo mejor modelo encontrado. Guardando...

      Fold 2: Entrenando...
 - Epoch 01: val_loss: 0.0959 - Epoch 02: val_loss: 0.0887 - Epoch 03: val_loss: 0.0823 - Epoch 04: val_loss: 0.0891 - Epoch 05: val_loss: 0.0877 - Epoch 06: val_loss: 0.0867 - Epoch 07: val_loss: 0.0831 - Epoch 08: val_loss: 0.0884 - Epoch 09: val_loss: 0.0928 - Epoch 10: val_loss: 0.1088 - Epoch 11: val_loss: 0.0968 - Epoch 12: val_loss: 0.0840 - Epoch 13: val_loss: 0.0944 - Epoch 14: val_loss: 0.0914

 - Epoch 15: val_loss: 0.0945
      Fold 2 completado. Mejor val_loss: 0.0823

      Fold 3: Entrenando...
 - Epoch 01: val_loss: 0.1021 - Epoch 02: val_loss: 0.0915 - Epoch 03: val_loss: 0.0909 - Epoch 04: val_loss: 0.0887 - Epoch 05: val_loss: 0.0808 - Epoch 06: val_loss: 0.0847 - Epoch 07: val_loss: 0.0790 - Epoch 08: val_loss: 0.0775 - Epoch 09: val_loss: 0.0867 - Epoch 10: val_loss: 0.0899 - Epoch 11: val_loss: 0.0883 - Epoch 12: val_loss: 0.0760 - Epoch 13: val_loss: 0.1167 - Epoch 14: val_loss: 0.1236 - Epoch 15: val_loss: 0.1582 - Epoch 16: val_loss: 0.1010 - Epoch 17: val_loss: 0.1492 - Epoch 18: val_loss: 0.1449 - Epoch 19: val_loss: 0.1315 - Epoch 20: val_loss: 0.1327 - Epoch 21: val_loss: 0.1629 - Epoch 22: val_loss: 0.1751 - Epoch 23: val_loss: 0.1606

 - Epoch 24: val_loss: 0.1821
      Fold 3 completado. Mejor val_loss: 0.0760
      -> Nuevo mejor modelo encontrado. Guardando...


Procesando Tareas LSTM: 100%|██████████| 222/222 [3:47:03<00:00, 61.37s/it]


Procesamiento completado. Generando resúmenes...

[OK] Métricas detalladas guardadas.
[OK] Resumen por ciudad y escenario guardado.
[OK] Resumen global guardado.
[OK] Resumen final guardado.

--- Resumen Final (promedio de todo) ---
       Imputation_Method  Final_Avg_MAE  Final_Avg_RMSE  Final_Avg_MASE
0              LSTM-VAE      14.961069       18.227050        9.752477
1  Linear Interpolation      14.701326       18.114916       11.751134
2      MOMENT Finetuned      14.179346       17.371398        9.330400
3     MOMENT Pretrained      14.293867       17.303082        9.123827
4         Random Forest      14.191684       17.400346       10.060825
5                SARIMA      14.383649       17.913854       10.911417

Script completado. Tiempo total: 227.06 minutos.


lstm solo con baseline

In [11]:
import pandas as pd
import numpy as np
import os
import random
import tensorflow as tf
from keras import layers, Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (20, 8), 'axes.titlesize': 22, 'axes.labelsize': 18,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 16,
    'lines.linewidth': 2.5
})

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

# --- PARÁMETROS ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_LAGS = 120
LSTM_UNITS, DROPOUT_RATE, LEARNING_RATE, CLIPNORM, EPOCHS, BATCH_SIZE, N_WFV_FOLDS, ES_PATIENCE, LR_PATIENCE = 64, 0.2, 0.001, 1.0, 40, 128, 5, 12, 5
HISTORY_LEN_FOR_PLOT = 300

# --- RUTAS ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'LSTM_full_FT'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')
MODELS_TEMP_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'temp_models_baseline')
os.makedirs(MODELS_TEMP_DIR, exist_ok=True)
os.makedirs(PLOTS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)

# --- FUNCIONES AUXILIARES ---
def create_sequences_single_shot(input_data, target_data, n_lags, horizon):
    X, y, indices = [], [], []
    if target_data.ndim == 1:
        target_data = target_data.reshape(-1, 1)
    
    end_idx = len(input_data) - n_lags - horizon + 1
    if end_idx <= 0: return np.array(X), np.array(y), np.array(indices)
    
    for i in range(end_idx):
        X.append(input_data[i:(i + n_lags)])
        y.append(target_data[i + n_lags : i + n_lags + horizon].flatten())
    
    X = np.array(X)
    y = np.array(y)
    indices = np.arange(n_lags, n_lags + len(X))
    return X, y, indices

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive_on_train) or mae_naive_on_train < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive_on_train
    return mase if np.isfinite(mase) else np.nan

def generate_lstm_forecasting_plots_single_shot(df_imputed_data, model, scaler_all, scaler_target, city, scenario, method, rep_id, output_dir, history_len=300):
    try:
        max_horizon = max(PREDICTION_HORIZONS)
        split_point = len(df_imputed_data) - history_len - max_horizon
        if split_point < N_LAGS: return
        all_cols = EXOGENOUS_COLS + TIME_FEATURES + [TARGET_VARIABLE]
        history_df = df_imputed_data.iloc[split_point - N_LAGS : split_point]
        input_seq = scaler_all.transform(history_df[all_cols])
        input_seq = input_seq.reshape(1, N_LAGS, input_seq.shape[1])
        prediction_scaled = model.predict(input_seq, verbose=0)
        prediction_unscaled = scaler_target.inverse_transform(prediction_scaled).flatten()
        
        for horizon in PREDICTION_HORIZONS:
            plt.figure(figsize=(20, 8))
            future_gt_df = df_imputed_data.iloc[split_point : split_point + horizon]
            future_pred = prediction_unscaled[:horizon]
            plot_history = df_imputed_data.iloc[split_point-history_len:split_point]
            plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
            plt.plot(future_gt_df.index, future_gt_df[TARGET_VARIABLE], label='Ground Truth', color='red')
            plt.plot(future_gt_df.index, future_pred, label='Forecast', color='green', linestyle='--')
            plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
            plt.title(f'Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
            plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
            plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
            scenario_safe = scenario.replace('/', '_').replace('\\', '_')
            plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe)
            os.makedirs(plot_save_dir, exist_ok=True)
            plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe}_{rep_id}_h{horizon}.png'
            plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
            plt.close()
    except Exception:
        plt.close()

def train_and_eval_baseline(df_city, target_col, all_features_cols, n_lags, horizons, plot_params):
    output_horizon = max(horizons)
    scaler_all = MinMaxScaler(); scaled_data = scaler_all.fit_transform(df_city[all_features_cols])
    scaler_target = MinMaxScaler(); scaled_target = scaler_target.fit_transform(df_city[[target_col]])
    train_val_size = int(len(scaled_data) * 0.8)
    train_val_data = scaled_data[:train_val_size]
    test_data = scaled_data[train_val_size:]
    tf.keras.backend.clear_session()
    X_tv, y_tv, _ = create_sequences_single_shot(train_val_data, scaled_target, n_lags, output_horizon)
    
    print(f"      - Tamaño del set de Train/Val para secuencias: {len(X_tv)}")
    if len(X_tv) < 200: return []
    
    tscv = TimeSeriesSplit(n_splits=N_WFV_FOLDS)
    best_val_loss = float('inf')
    model_path = os.path.join(MODELS_TEMP_DIR, f"best_model_baseline_{plot_params['city'].replace(' ', '_')}.keras")
    
    for fold, (train_idx, val_idx) in enumerate(tqdm(tscv.split(X_tv), total=N_WFV_FOLDS, desc="    WFV Folds", leave=False)):
        X_train, y_train = X_tv[train_idx], y_tv[train_idx]
        X_val, y_val = X_tv[val_idx], y_tv[val_idx]
        
        inputs = layers.Input(shape=(X_train.shape[1], X_train.shape[2])); lstm_1 = layers.LSTM(LSTM_UNITS, return_sequences=True)(inputs); lstm_1 = layers.Dropout(DROPOUT_RATE)(lstm_1)
        lstm_2 = layers.LSTM(LSTM_UNITS // 2)(lstm_1); lstm_2 = layers.Dropout(DROPOUT_RATE)(lstm_2); outputs = layers.Dense(output_horizon)(lstm_2)
        model = Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=CLIPNORM)
        model.compile(optimizer=optimizer, loss='mse')
        es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=ES_PATIENCE, restore_best_weights=True)
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[es], verbose=0)
        
        val_loss = min(history.history['val_loss'])
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model.save(model_path)
    
    if not os.path.exists(model_path): return []
    best_model = tf.keras.models.load_model(model_path)
    
    # --- LLAMADA A PLOTEO CORREGIDA ---
    generate_lstm_forecasting_plots_single_shot(df_city, best_model, scaler_all, scaler_target, **plot_params)
        
    X_test, y_test, _ = create_sequences_single_shot(test_data, scaled_target.flatten(), n_lags, output_horizon)
    if len(X_test) == 0: return []
    
    predictions_scaled = best_model.predict(X_test, verbose=0)
    
    results_by_horizon = []
    y_train_unscaled = scaler_target.inverse_transform(y_tv[train_idx])
    for h in horizons:
        pred_h_unscaled = scaler_target.inverse_transform(predictions_scaled[:, :h])
        true_h_unscaled = scaler_target.inverse_transform(y_test[:, :h])
        mae = np.mean(np.abs(true_h_unscaled - pred_h_unscaled))
        rmse = np.sqrt(np.mean((true_h_unscaled - pred_h_unscaled)**2))
        mase = calculate_mase(true_h_unscaled.flatten(), pred_h_unscaled.flatten(), y_train_unscaled.flatten())
        results_by_horizon.append({'Horizon': h, 'MAE': mae, 'RMSE': rmse, 'MASE': mase})
        
    return results_by_horizon

# --- SCRIPT PRINCIPAL DE ACTUALIZACIÓN ---
if __name__ == '__main__':
    all_baseline_metrics = []
    try:
        df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
        if 'CO2' in df_original_full.columns:
            df_original_full = df_original_full.drop(columns=['CO2'])
        df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
        for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
            if col in df_original_full.columns:
                df_original_full[col] = pd.to_numeric(df_original_full[col], errors='coerce').ffill().bfill()
        df_original_full['Hour'] = df_original_full['Date'].dt.hour
        df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
        df_original_full['Month'] = df_original_full['Date'].dt.month
    except FileNotFoundError:
        print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'."); exit()

    print(f"\n{'='*20} Procesando únicamente: Original_Data_Baseline {'='*20}")
    for city_name in df_original_full['City'].unique():
        print(f"\n--- Procesando Baseline para ciudad: {city_name} ---")
        df_city_input = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_input.empty or df_city_input.isnull().values.any(): 
            print(f"  -> Datos para {city_name} vacíos o con NaNs irreparables. Omitiendo.")
            continue
        
        # --- DICCIONARIO DE PLOTEO CORREGIDO ---
        plot_params = {'city': city_name, 'scenario': 'N/A', 'method': 'Original_Data_Baseline', 'rep_id': 'baseline', 'output_dir': PLOTS_OUTPUT_DIR}
        
        horizon_metrics = train_and_eval_baseline(df_city_input, TARGET_VARIABLE, EXOGENOUS_COLS + TIME_FEATURES + [TARGET_VARIABLE], N_LAGS, PREDICTION_HORIZONS, plot_params)
        for hm in horizon_metrics:
            all_baseline_metrics.append({'Imputation_Method': 'Original_Data_Baseline', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']})
    
    print("\nProcesamiento del Baseline completado.")

    detailed_results_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    try:
        df_existing_metrics = pd.read_csv(detailed_results_path)
        print(f"\nResultados existentes cargados desde: {detailed_results_path}")
        df_existing_metrics = df_existing_metrics[df_existing_metrics['Imputation_Method'] != 'Original_Data_Baseline']
    except FileNotFoundError:
        print("No se encontró archivo de métricas existente. Se creará uno nuevo.")
        df_existing_metrics = pd.DataFrame()

    if not all_baseline_metrics:
        print("\nADVERTENCIA: No se generaron nuevos resultados del baseline. El proceso termina aquí.")
    else:
        df_baseline_metrics = pd.DataFrame(all_baseline_metrics)
        df_all_metrics = pd.concat([df_existing_metrics, df_baseline_metrics], ignore_index=True)
        print("\n--- Regenerando todos los archivos de resultados ---")
        df_all_metrics.to_csv(detailed_results_path, index=False)
        print(f"[OK] Archivo de métricas detalladas actualizado: {detailed_results_path}")
        
        df_summary = df_all_metrics.groupby(['Imputation_Method', 'City', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index()
        df_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_and_scenario_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen por ciudad y escenario regenerado.")
        
        df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        df_global_summary.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen global regenerado.")
        print("\n--- Resumen Global por Método y Horizonte (Actualizado) ---\n", df_global_summary.sort_values(by=['Imputation_Method', 'Horizon_h']).to_string())
        
        df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        df_avg_over_horizons.to_csv(os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv'), index=False)
        print("[OK] Resumen final regenerado.")
        print("\n--- Resumen Final por Método (Actualizado) ---\n", df_avg_over_horizons.sort_values(by='Final_Avg_MAE').to_string())

    print("\n--- Proceso de actualización completado. ---")


==================== Procesando únicamente: Original_Data_Baseline ====================

--- Procesando Baseline para ciudad: Brasilia ---
      - Tamaño del set de Train/Val para secuencias: 6848



--- Procesando Baseline para ciudad: Cairo ---
      - Tamaño del set de Train/Val para secuencias: 6848



--- Procesando Baseline para ciudad: Dubai ---
      - Tamaño del set de Train/Val para secuencias: 6848



--- Procesando Baseline para ciudad: London ---
      - Tamaño del set de Train/Val para secuencias: 6848



--- Procesando Baseline para ciudad: New York ---
      - Tamaño del set de Train/Val para secuencias: 6848



--- Procesando Baseline para ciudad: Sydney ---
      - Tamaño del set de Train/Val para secuencias: 6848



Procesamiento del Baseline completado.

Resultados existentes cargados desde: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\predicciones\LSTM_full_FT\metricas\ALL_prediction_metrics_detailed_LSTM_full_FT.csv

--- Regenerando todos los archivos de resultados ---
[OK] Archivo de métricas detalladas actualizado: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\predicciones\LSTM_full_FT\metricas\ALL_prediction_metrics_detailed_LSTM_full_FT.csv
[OK] Resumen por ciudad y escenario regenerado.
[OK] Resumen global regenerado.

--- Resumen Global por Método y Horizonte (Actualizado) ---
          Imputation_Method  Horizon_h  Global_Avg_MAE  Global_Avg_RMSE  Global_Avg_MASE
0                 LSTM-VAE         12       14.456307        17.917920         9.441740
1                 LSTM-VAE         24       14.872390        18.168479         9.703178
2                 LSTM-VAE         36       15.032385        18.257381         9.804651
3                 LSTM-VAE      

MOMENT_PRE UNIVARIADO

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
from momentfm import MOMENTPipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import random
import traceback

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (20, 8), 'axes.titlesize': 22, 'axes.labelsize': 18,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 16,
    'lines.linewidth': 2.5
})

# --- CONFIGURACIÓN DE GPU ---
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == 'cuda':
    try:
        torch.cuda.set_device(0)
        print(f"Usando dispositivo: {device} ({torch.cuda.get_device_name(0)})")
    except Exception as e:
        print(f"Error configurando GPU, usando CPU. Error: {e}"); device = "cpu"
else:
    print(f"Usando dispositivo: {device}")

# --- PARÁMETROS ---
TARGET_VARIABLE = 'AQI'
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
MAX_HORIZON = max(PREDICTION_HORIZONS)
CONTEXT_LENGTH = 512
N_CV_SPLITS_PREDICT = 3
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Pretrained', 'MOMENT Finetuned', 'Random Forest', 'SARIMA'
]
MODEL_NAME = "AutonLab/MOMENT-1-large"
N_REPETITIONS_TO_PROCESS = 1

# --- RUTAS ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_PRE_full_FT' # Nombre actualizado
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')
PREDICTION_CSVS_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'predictions_csvs')
for d in [PREDICTION_METRICS_OUTPUT_DIR, PREDICTION_PLOTS_OUTPUT_DIR, PREDICTION_CSVS_DIR]:
    os.makedirs(d, exist_ok=True)


# --- CLASES Y FUNCIONES ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive) or mae_naive < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive
    return mase if np.isfinite(mase) else np.nan

def save_prediction_results_to_csv(context_data, ground_truth_data, prediction_data, date_index, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True) # Asegurar que el subdirectorio exista
    df_pred = pd.DataFrame({'date': date_index.values, 'context': np.nan, 'ground_truth': np.nan, 'prediction': np.nan})
    context_len, horizon_len = len(context_data), len(ground_truth_data)
    df_pred.loc[:context_len-1, 'context'] = context_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'ground_truth'] = ground_truth_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'prediction'] = prediction_data
    df_pred.to_csv(file_path, index=False)

def plot_and_save_forecast(context, ground_truth, prediction, date_index, title, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True) # Asegurar que el subdirectorio exista
    plt.figure(figsize=(18, 7))
    context_dates = date_index[:len(context)]
    forecast_dates = date_index[len(context):len(context)+len(prediction)]
    plt.plot(context_dates, context, label='Contexto (Datos de Entrada)', color='royalblue')
    plt.plot(forecast_dates, ground_truth, label='Valores Reales (Ground Truth)', color='red')
    plt.plot(forecast_dates, prediction, label='Predicción del Modelo', color='green', linestyle='--')
    
    if not forecast_dates.empty:
        plt.axvline(x=forecast_dates.iloc[0], color='black', linestyle='--', linewidth=1.5, label='Inicio Predicción')
        
    plt.title(title, fontsize=16)
    plt.xlabel('Fecha')
    plt.ylabel(f'Valor de {TARGET_VARIABLE}')
    plt.legend(); plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout(); plt.savefig(file_path, dpi=100); plt.close()

# --- FUNCIÓN DE EVALUACIÓN RÁPIDA (CV) ---
def predict_and_evaluate_moment_cv(df_city_data, target_col, model_pipeline, device, scaler, **kwargs):
    model_pipeline.eval()
    series_data = df_city_data[target_col]
    min_len = CONTEXT_LENGTH + MAX_HORIZON + N_CV_SPLITS_PREDICT 
    if len(series_data) < min_len:
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in PREDICTION_HORIZONS}
    
    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS_PREDICT, test_size=MAX_HORIZON)
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS}
    
    last_fold_data = {}
    for fold, (train_idx, test_idx) in enumerate(tscv.split(series_data)):
        train_series = series_data.iloc[train_idx]
        context_unscaled = train_series.tail(CONTEXT_LENGTH).values
        if len(context_unscaled) < CONTEXT_LENGTH: continue
        
        context_scaled = scaler.transform(context_unscaled.reshape(-1, 1)).flatten()
        context_tensor = torch.tensor(context_scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            try:
                outputs = model_pipeline(x_enc=context_tensor, forecast_len=MAX_HORIZON)
                predictions_scaled = np.atleast_1d(outputs.forecast.cpu().squeeze().numpy())
            except Exception: continue
        
        ground_truth_unscaled = series_data.iloc[test_idx].values
        predictions_unscaled = scaler.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()

        for h in PREDICTION_HORIZONS:
            y_pred_h, y_true_h = predictions_unscaled[h-1], ground_truth_unscaled[h-1]
            accumulated_metrics[h]['maes'].append(np.abs(y_true_h - y_pred_h))
            accumulated_metrics[h]['rmses'].append((y_true_h - y_pred_h)**2)
            mase = calculate_mase(np.array([y_true_h]), np.array([y_pred_h]), train_series)
            if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)
        
        if fold == N_CV_SPLITS_PREDICT - 1:
            start_idx, end_idx = train_idx[-CONTEXT_LENGTH], test_idx[-1]
            last_fold_data = {'context': context_unscaled, 'ground_truth': ground_truth_unscaled, 'prediction': predictions_unscaled, 'date_index': df_city_data['Date'].iloc[start_idx:end_idx+1]}

    if last_fold_data and 'plot_path' in kwargs and 'csv_path' in kwargs:
        plot_and_save_forecast(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], last_fold_data['date_index'], kwargs['plot_title'], kwargs['plot_path'])
        save_prediction_results_to_csv(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], last_fold_data['date_index'], kwargs['csv_path'])

    results = {}
    for h in PREDICTION_HORIZONS:
        results[h] = {'MAE': np.nanmean(accumulated_metrics[h]['maes']), 'RMSE': np.sqrt(np.nanmean(accumulated_metrics[h]['rmses'])), 'MASE': np.nanmean(accumulated_metrics[h]['mases'])}
    return results

# --- SCRIPT PRINCIPAL ---
if __name__ == '__main__':
    all_prediction_metrics_detailed = []
    script_start_time = time.time()
    try:
        moment_predictor = MOMENTPipeline.from_pretrained(
            MODEL_NAME, 
            model_kwargs={'task_name': 'forecasting', 'forecast_horizon': MAX_HORIZON}
        )
        moment_predictor.init(); moment_predictor = moment_predictor.to(device)
        print("Modelo MOMENT para predicción cargado e inicializado.")
        
        df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
        df_original_full['Date'] = pd.to_datetime(df_original_full['Date'], utc=True)
        if 'CO2' in df_original_full.columns: df_original_full = df_original_full.drop(columns=['CO2'])
        print(f"Datos originales '{ORIGINAL_DATA_FILE}' cargados.")
    except Exception as e:
        print(f"Error crítico en la inicialización: {e}"); exit()

    cities_to_process = df_original_full['City'].unique()
    
    print("\n--- Procesando Baseline ---")
    method_name_baseline = "Original_Data_Baseline"
    for city_name in tqdm(cities_to_process, desc="Procesando Ciudades (Baseline)"):
        df_city_data = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        df_city_data[TARGET_VARIABLE] = pd.to_numeric(df_city_data[TARGET_VARIABLE], errors='coerce').ffill().bfill()
        if df_city_data.empty or df_city_data[TARGET_VARIABLE].isnull().any(): continue
        
        scaler = StandardScaler().fit(df_city_data[[TARGET_VARIABLE]])
        city_safe_name = city_name.replace(" ", "_")
        
        # --- NUEVA ESTRUCTURA DE RUTAS para Baseline ---
        plot_title = f'MOMENT-PRE Predicción Baseline - {city_name}'
        plot_path = os.path.join(PREDICTION_PLOTS_OUTPUT_DIR, method_name_baseline, city_safe_name, f'baseline_{city_safe_name}.png')
        csv_path = os.path.join(PREDICTION_CSVS_DIR, method_name_baseline, city_safe_name, f'baseline_{city_safe_name}.csv')

        horizon_metrics = predict_and_evaluate_moment_cv(df_city_data, TARGET_VARIABLE, moment_predictor, device, scaler, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
        for h, metrics in horizon_metrics.items():
            all_prediction_metrics_detailed.append({'Imputation_Method': method_name_baseline, 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})

    print("\n--- Procesando Datos Imputados ---")
    for method_name in METHODS_TO_PROCESS:
        print(f"\n--- Método de Imputación: {method_name} ---")
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue

        city_folders_list = [d for d in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, d))]
        for city_folder in tqdm(city_folders_list, desc=f"Ciudades en {method_name}", leave=True):
            city_path = os.path.join(imputed_csvs_root, city_folder)
            city_name = city_folder.replace("_", " ")
            if city_name not in cities_to_process: continue
            
            df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
            scaler = StandardScaler().fit(df_city_ground_truth[[TARGET_VARIABLE]])

            scenario_folders_list = [d for d in os.listdir(city_path) if os.path.isdir(os.path.join(city_path, d))]
            for scenario_folder in tqdm(scenario_folders_list, desc=f"  Escenarios en {city_name}", leave=False):
                scenario_path = os.path.join(city_path, scenario_folder)
                
                files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
                
                for rep_file_path in files_to_process:
                    rep_id = os.path.basename(rep_file_path)
                    try:
                        df_rep = pd.read_csv(rep_file_path)
                        df_rep['Date'] = pd.to_datetime(df_rep['Date'], utc=True)
                        if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                        if df_rep.empty or df_rep[TARGET_VARIABLE].isnull().any(): continue
                    except Exception: continue
                    
                    # --- NUEVA ESTRUCTURA DE RUTAS para Datos Imputados ---
                    city_safe_name = city_name.replace(" ", "_")
                    safe_rep_id = rep_id.replace('.csv', '') # Crear un id seguro para el nombre de archivo
                    plot_title = f'MOMENT-PRE - {city_name}\nMétodo: {method_name}\nEscenario: {scenario_folder}'
                    
                    # Crear subdirectorios anidados
                    plot_dir = os.path.join(PREDICTION_PLOTS_OUTPUT_DIR, method_name, city_safe_name, scenario_folder)
                    csv_dir = os.path.join(PREDICTION_CSVS_DIR, method_name, city_safe_name, scenario_folder)
                    
                    plot_path = os.path.join(plot_dir, f'{safe_rep_id}.png')
                    csv_path = os.path.join(csv_dir, f'{safe_rep_id}.csv')

                    horizon_metrics = predict_and_evaluate_moment_cv(df_rep, TARGET_VARIABLE, moment_predictor, device, scaler, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
                    for h, metrics in horizon_metrics.items():
                        all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

    # Guardado de Métricas
    if all_prediction_metrics_detailed:
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        detailed_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.to_csv(detailed_path, index=False)
        print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")
        
        summary_city_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_and_scenario_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
        print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")
        
        global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        global_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
        global_summary.to_csv(global_summary_path, index=False)
        print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")
        
        final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        final_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
        final_summary.to_csv(final_summary_path, index=False)
        print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
        print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary.sort_values(by='Final_Avg_MAE'))
    else:
        print("\nNo se generaron resultados.")
        
    print(f"\nScript completado. Tiempo total: {(time.time() - script_start_time)/60:.2f} minutos.")

Usando dispositivo: cuda (NVIDIA GeForce RTX 3060 Laptop GPU)
Modelo MOMENT para predicción cargado e inicializado.
Datos originales 'c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargados.

--- Procesando Baseline ---


Procesando Ciudades (Baseline): 100%|██████████| 6/6 [00:02<00:00,  2.08it/s]



--- Procesando Datos Imputados ---

--- Método de Imputación: Linear Interpolation ---


Ciudades en Linear Interpolation:  50%|█████     | 3/6 [00:11<00:11,  3.88s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
from momentfm import MOMENTPipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import random
import traceback

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (20, 8), 'axes.titlesize': 22, 'axes.labelsize': 18,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 16,
    'lines.linewidth': 2.5
})

# --- CONFIGURACIÓN DE GPU ---
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == 'cuda':
    try:
        torch.cuda.set_device(0)
        print(f"Usando dispositivo: {device} ({torch.cuda.get_device_name(0)})")
    except Exception as e:
        print(f"Error configurando GPU, usando CPU. Error: {e}"); device = "cpu"
else:
    print(f"Usando dispositivo: {device}")

# --- PARÁMETROS ---
TARGET_VARIABLE = 'AQI'
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
MAX_HORIZON = max(PREDICTION_HORIZONS)
CONTEXT_LENGTH = 512
N_CV_SPLITS_PREDICT = 3
METHODS_TO_PROCESS = [
    'Linear Interpolation', 'LSTM-VAE', 'MOMENT Pretrained', 'MOMENT Finetuned', 'Random Forest', 'SARIMA'
]
MODEL_NAME = "AutonLab/MOMENT-1-large"
N_REPETITIONS_TO_PROCESS = 1

# --- RUTAS ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_PRE_Pred_Univar_CV_Fast_Hierarchical'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')
PREDICTION_CSVS_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'predictions_csvs')
for d in [PREDICTION_METRICS_OUTPUT_DIR, PREDICTION_PLOTS_OUTPUT_DIR, PREDICTION_CSVS_DIR]:
    os.makedirs(d, exist_ok=True)


# --- CLASES Y FUNCIONES ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    with np.errstate(divide='ignore', invalid='ignore'):
        mae_naive = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
        if not np.isfinite(mae_naive) or mae_naive < 1e-9: return np.nan
        mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
        mase = mae_forecast / mae_naive
    return mase if np.isfinite(mase) else np.nan

def save_prediction_results_to_csv(context_data, ground_truth_data, prediction_data, date_index, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    df_pred = pd.DataFrame({'date': date_index.values, 'context': np.nan, 'ground_truth': np.nan, 'prediction': np.nan})
    context_len, horizon_len = len(context_data), len(ground_truth_data)
    df_pred.loc[:context_len-1, 'context'] = context_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'ground_truth'] = ground_truth_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'prediction'] = prediction_data
    df_pred.to_csv(file_path, index=False)

def plot_and_save_forecast(context, ground_truth, prediction, date_index, title, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    plt.figure(figsize=(18, 7))
    context_dates = date_index[:len(context)]
    forecast_dates = date_index[len(context):len(context)+len(prediction)]
    plt.plot(context_dates, context, label='Observed History', color='royalblue')
    plt.plot(forecast_dates, ground_truth, label='Ground Truth', color='red')
    plt.plot(forecast_dates, prediction, label='Forecast', color='green', linestyle='--')
    if not forecast_dates.empty:
        plt.axvline(x=forecast_dates.iloc[0], color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
    plt.title(title, fontsize=16)
    plt.xlabel('Date')
    plt.ylabel(f'{TARGET_VARIABLE} Value')
    plt.legend(loc='upper left'); plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout(); plt.savefig(file_path, dpi=100); plt.close()

# --- FUNCIÓN DE EVALUACIÓN CORREGIDA ---
def predict_and_evaluate_moment_cv(df_input_data, df_ground_truth, target_col, model_pipeline, device, scaler, **kwargs):
    model_pipeline.eval()
    # Los datos de entrada para la predicción provienen del df imputado/procesado
    series_data_input = df_input_data[target_col]
    # El ground truth para la evaluación SIEMPRE proviene del df original limpio
    series_data_ground_truth = df_ground_truth[target_col]

    min_len = CONTEXT_LENGTH + MAX_HORIZON + N_CV_SPLITS_PREDICT 
    if len(series_data_input) < min_len:
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in PREDICTION_HORIZONS}
    
    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS_PREDICT, test_size=MAX_HORIZON)
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS}
    
    last_fold_data = {}
    # La división de CV se hace sobre el índice de los datos de entrada
    for fold, (train_idx, test_idx) in enumerate(tscv.split(series_data_input)):
        train_series = series_data_input.iloc[train_idx]
        context_unscaled = train_series.tail(CONTEXT_LENGTH).values
        if len(context_unscaled) < CONTEXT_LENGTH: continue
        
        context_scaled = scaler.transform(context_unscaled.reshape(-1, 1)).flatten()
        context_tensor = torch.tensor(context_scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            try:
                outputs = model_pipeline(x_enc=context_tensor, forecast_len=MAX_HORIZON)
                predictions_scaled = np.atleast_1d(outputs.forecast.cpu().squeeze().numpy())
            except Exception: continue
        
        # --- CORRECCIÓN: Usar siempre el ground truth original ---
        ground_truth_unscaled = series_data_ground_truth.iloc[test_idx].values
        predictions_unscaled = scaler.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()

        for h in PREDICTION_HORIZONS:
            y_pred_h, y_true_h = predictions_unscaled[h-1], ground_truth_unscaled[h-1]
            accumulated_metrics[h]['maes'].append(np.abs(y_true_h - y_pred_h))
            accumulated_metrics[h]['rmses'].append((y_true_h - y_pred_h)**2)
            mase = calculate_mase(np.array([y_true_h]), np.array([y_pred_h]), train_series)
            if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)
        
        if fold == N_CV_SPLITS_PREDICT - 1:
            start_idx, end_idx = test_idx[0] - CONTEXT_LENGTH, test_idx[-1]
            last_fold_data = {
                'context': context_unscaled, 
                'ground_truth': ground_truth_unscaled, 
                'prediction': predictions_unscaled, 
                # Las fechas deben corresponder a la ventana completa de contexto + predicción
                'date_index': df_input_data['Date'].iloc[start_idx:end_idx+1]
            }

    if last_fold_data and 'plot_path' in kwargs and 'csv_path' in kwargs:
        plot_and_save_forecast(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], last_fold_data['date_index'], kwargs['plot_title'], kwargs['plot_path'])
        save_prediction_results_to_csv(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], last_fold_data['date_index'], kwargs['csv_path'])

    results = {}
    for h in PREDICTION_HORIZONS:
        results[h] = {'MAE': np.nanmean(accumulated_metrics[h]['maes']), 'RMSE': np.sqrt(np.nanmean(accumulated_metrics[h]['rmses'])), 'MASE': np.nanmean(accumulated_metrics[h]['mases'])}
    return results

# --- SCRIPT PRINCIPAL ---
if __name__ == '__main__':
    all_prediction_metrics_detailed = []
    script_start_time = time.time()
    try:
        moment_predictor = MOMENTPipeline.from_pretrained(
            MODEL_NAME, 
            model_kwargs={'task_name': 'forecasting', 'forecast_horizon': MAX_HORIZON}
        )
        moment_predictor.init(); moment_predictor = moment_predictor.to(device)
        print("Modelo MOMENT para predicción cargado e inicializado.")
        
        df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
        df_original_full['Date'] = pd.to_datetime(df_original_full['Date'], utc=True)
        # Crear una versión limpia para usarla siempre como ground truth
        df_clean_original_full = df_original_full.copy()
        df_clean_original_full[TARGET_VARIABLE] = df_clean_original_full[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
        print(f"Datos originales '{ORIGINAL_DATA_FILE}' cargados y limpiados para ground truth.")
    except Exception as e:
        print(f"Error crítico en la inicialización: {e}"); exit()

    cities_to_process = df_clean_original_full['City'].unique()
    
    print("\n--- Procesando Baseline ---")
    method_name_baseline = "Original_Data_Baseline"
    for city_name in tqdm(cities_to_process, desc="Procesando Ciudades (Baseline)"):
        df_city_data = df_clean_original_full[df_clean_original_full['City'] == city_name].copy().reset_index(drop=True)
        if df_city_data.empty: continue
        
        scaler = StandardScaler().fit(df_city_data[[TARGET_VARIABLE]])
        city_safe_name = city_name.replace(" ", "_")
        
        plot_title = f'MOMENT-PRE Forecast in {city_name}\nImputation: {method_name_baseline} | Scenario: N/A'
        plot_path = os.path.join(PREDICTION_PLOTS_OUTPUT_DIR, method_name_baseline, city_safe_name, f'forecast_{city_safe_name}.png')
        csv_path = os.path.join(PREDICTION_CSVS_DIR, method_name_baseline, city_safe_name, f'forecast_{city_safe_name}.csv')
        
        # Para el baseline, los datos de entrada y el ground truth son los mismos (los datos limpios)
        horizon_metrics = predict_and_evaluate_moment_cv(df_city_data, df_city_data, TARGET_VARIABLE, moment_predictor, device, scaler, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
        for h, metrics in horizon_metrics.items():
            all_prediction_metrics_detailed.append({'Imputation_Method': method_name_baseline, 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})

    print("\n--- Procesando Datos Imputados ---")
    for method_name in METHODS_TO_PROCESS:
        print(f"\n--- Método de Imputación: {method_name} ---")
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue

        city_folders_list = [d for d in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, d))]
        for city_folder in tqdm(city_folders_list, desc=f"Ciudades en {method_name}", leave=True):
            city_path = os.path.join(imputed_csvs_root, city_folder)
            city_name = city_folder.replace("_", " ")
            if city_name not in cities_to_process: continue
            
            # El ground truth es SIEMPRE el dataframe original limpio para esa ciudad
            df_city_ground_truth = df_clean_original_full[df_clean_original_full['City'] == city_name].copy().reset_index(drop=True)
            scaler = StandardScaler().fit(df_city_ground_truth[[TARGET_VARIABLE]])

            scenario_folders_list = [d for d in os.listdir(city_path) if os.path.isdir(os.path.join(city_path, d))]
            for scenario_folder in tqdm(scenario_folders_list, desc=f"  Escenarios en {city_name}", leave=False):
                scenario_path = os.path.join(city_path, scenario_folder)
                
                files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
                
                for rep_file_path in files_to_process:
                    rep_id = os.path.basename(rep_file_path)
                    try:
                        df_rep = pd.read_csv(rep_file_path)
                        df_rep['Date'] = pd.to_datetime(df_rep['Date'], utc=True)
                        if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                        if df_rep.empty: continue
                    except Exception: continue
                    
                    city_safe_name = city_name.replace(" ", "_")
                    safe_rep_id = rep_id.replace('.csv', '')
                    
                    plot_title = f'MOMENT-PRE Forecast in {city_name}\nImputation: {method_name} | Scenario: {scenario_folder}'
                    plot_dir = os.path.join(PREDICTION_PLOTS_OUTPUT_DIR, method_name, city_safe_name, scenario_folder)
                    csv_dir = os.path.join(PREDICTION_CSVS_DIR, method_name, city_safe_name, scenario_folder)
                    plot_path = os.path.join(plot_dir, f'{safe_rep_id}.png')
                    csv_path = os.path.join(csv_dir, f'{safe_rep_id}.csv')

                    # Pasamos el df imputado (df_rep) y el df original limpio (df_city_ground_truth)
                    horizon_metrics = predict_and_evaluate_moment_cv(df_rep, df_city_ground_truth, TARGET_VARIABLE, moment_predictor, device, scaler, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
                    for h, metrics in horizon_metrics.items():
                        all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

    # Guardado de Métricas... (sin cambios)
    if all_prediction_metrics_detailed:
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        detailed_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.to_csv(detailed_path, index=False)
        print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")
        
        summary_city_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'summary_by_city_and_scenario_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
        print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")
        
        global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        global_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
        global_summary.to_csv(global_summary_path, index=False)
        print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")
        
        final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        final_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
        final_summary.to_csv(final_summary_path, index=False)
        print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
        print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary.sort_values(by='Final_Avg_MAE'))
    else:
        print("\nNo se generaron resultados.")
        
    print(f"\nScript completado. Tiempo total: {(time.time() - script_start_time)/60:.2f} minutos.")

Usando dispositivo: cuda (NVIDIA GeForce RTX 3060 Laptop GPU)
Modelo MOMENT para predicción cargado e inicializado.
Datos originales 'c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargados y limpiados para ground truth.

--- Procesando Baseline ---


Procesando Ciudades (Baseline): 100%|██████████| 6/6 [00:02<00:00,  2.15it/s]



--- Procesando Datos Imputados ---

--- Método de Imputación: Linear Interpolation ---


Ciudades en Linear Interpolation: 100%|██████████| 6/6 [00:19<00:00,  3.25s/it]



--- Método de Imputación: LSTM-VAE ---


...


--- Método de Imputación: Random Forest ---


Ciudades en Random Forest: 100%|██████████| 6/6 [00:16<00:00,  2.79s/it]



--- Método de Imputación: SARIMA ---


Ciudades en SARIMA: 100%|██████████| 6/6 [00:17<00:00,  2.99s/it]


[OK] Métricas detalladas guardadas en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\predicciones\MOMENT_PRE_Pred_Univar_CV_Fast_Hierarchical\metricas\ALL_prediction_metrics_detailed_MOMENT_PRE_Pred_Univar_CV_Fast_Hierarchical.csv
[OK] Resumen por ciudad y escenario guardado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\predicciones\MOMENT_PRE_Pred_Univar_CV_Fast_Hierarchical\metricas\summary_by_city_and_scenario_MOMENT_PRE_Pred_Univar_CV_Fast_Hierarchical.csv
[OK] Resumen global por método y horizonte guardado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\predicciones\MOMENT_PRE_Pred_Univar_CV_Fast_Hierarchical\metricas\global_summary_MOMENT_PRE_Pred_Univar_CV_Fast_Hierarchical.csv

[OK] Resumen final promediando horizontes guardado en: c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\predicciones\MOMENT_PRE_Pred_Univar_CV_Fast_Hierarchical\metricas\final_summary_avg_over_horizons_MOMENT_PRE_Pred_Univar_CV_Fast_Hier

MOMENT_FT UNIVARIADO

In [1]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import traceback
import time
import glob
import warnings
import joblib
import sys
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# --- 1. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
MODEL_NAME = "AutonLab/MOMENT-1-base"
CONTEXT_LENGTH = 512
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
MAX_HORIZON = max(PREDICTION_HORIZONS)

# --- Parámetros de Fine-Tuning ---
FINETUNE_EPOCHS = 1
BATCH_SIZE_FINETUNE = 32
LR = 1e-4
FINETUNE_TRAIN_SPLIT_RATIO = 0.8
ES_PATIENCE_FT = 5
MIN_DELTA_FT = 1e-5

# --- Parámetros de Predicción ---
N_CV_SPLITS_PREDICT = 2

# --- Filtros para Pruebas ---
TARGET_CITY_FILTER = None
N_REPETITIONS_TO_PROCESS = 1

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_FT_with_Plots_CorrectedGT' # Nuevo nombre
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
MODELS_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'modelos')
SCALERS_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'scalers')
PREDICTION_PLOTS_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')
PREDICTION_CSVS_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'predictions_csvs')
for d in [PREDICTION_METRICS_ROOT_DIR, MODELS_DIR, SCALERS_DIR, PREDICTION_PLOTS_DIR, PREDICTION_CSVS_DIR]:
    os.makedirs(d, exist_ok=True)


# --- 2. Clases y Funciones ---
class MomentForecastingDataset(Dataset):
    def __init__(self, data_array, context_length, max_horizon):
        self.data_array, self.context_length, self.max_horizon = data_array, context_length, max_horizon
        self.n_samples = max(0, len(data_array) - context_length - max_horizon + 1)
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        context = self.data_array[idx : idx + self.context_length]
        target = self.data_array[idx + self.context_length : idx + self.context_length + self.max_horizon]
        return {'context': torch.tensor(context, dtype=torch.float32), 'ground_truth_forecast': torch.tensor(target, dtype=torch.float32)}

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive < 1e-9: return np.nan
    return np.mean(np.abs(y_true - y_pred)) / mae_naive

def train_moment_forecasting_model(model, train_loader, device, epochs, variable_name, model_save_path, max_horizon, learning_rate):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    best_loss = float('inf')
    epochs_no_improve = 0
    print(f"  -> Iniciando fine-tuning para '{variable_name}' en {device.upper()} ({epochs} épocas máx)...")
    for epoch in range(epochs):
        total_epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"    Epoch {epoch+1}/{epochs}", leave=False)
        for batch_data in pbar:
            context, ground_truth = batch_data['context'].unsqueeze(1).to(device), batch_data['ground_truth_forecast'].unsqueeze(1).to(device)
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=context, forecast_len=max_horizon)
                loss = criterion(outputs.forecast, ground_truth)
                if torch.isnan(loss): continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                total_epoch_loss += loss.item()
                pbar.set_postfix(loss=loss.item())
            except Exception: continue
        avg_epoch_loss = total_epoch_loss / len(train_loader) if len(train_loader) > 0 else float('inf')
        print(f"    Epoch {epoch+1} Loss: {avg_epoch_loss:.6f}")
        if avg_epoch_loss < best_loss - MIN_DELTA_FT:
            best_loss, epochs_no_improve = avg_epoch_loss, 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= ES_PATIENCE_FT:
            print(f"    -> Early stopping en epoch {epoch+1}."); break
    print(f"  -> Fine-tuning completado. Mejor pérdida: {best_loss:.6f}")
    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

def save_prediction_results_to_csv(context_data, ground_truth_data, prediction_data, date_index, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    df_pred = pd.DataFrame({'date': date_index.values, 'context': np.nan, 'ground_truth': np.nan, 'prediction': np.nan})
    context_len, horizon_len = len(context_data), len(ground_truth_data)
    df_pred.loc[:context_len-1, 'context'] = context_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'ground_truth'] = ground_truth_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'prediction'] = prediction_data
    df_pred.to_csv(file_path, index=False)

def plot_and_save_forecast(context, ground_truth, prediction, date_index, title, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    plt.figure(figsize=(18, 7))
    context_dates = date_index[:len(context)]
    forecast_dates = date_index[len(context):len(context)+len(prediction)]
    plt.plot(context_dates, context, label='Observed History', color='royalblue')
    plt.plot(forecast_dates, ground_truth, label='Ground Truth', color='red')
    plt.plot(forecast_dates, prediction, label='Forecast', color='green', linestyle='--')
    if not forecast_dates.empty:
        plt.axvline(x=forecast_dates.iloc[0], color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
    plt.title(title, fontsize=16)
    plt.xlabel('Date'); plt.ylabel(f'{TARGET_VARIABLE} Value')
    plt.legend(loc='upper left'); plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout(); plt.savefig(file_path, dpi=100); plt.close()

# --- FUNCIÓN DE EVALUACIÓN CORREGIDA ---
def predict_and_evaluate_moment_cv(df_input_data, df_ground_truth, target_col, model_pipeline, device, scaler, **kwargs):
    model_pipeline.eval()
    series_data_input = df_input_data[target_col]
    series_data_ground_truth = df_ground_truth[target_col]
    
    min_len = CONTEXT_LENGTH + MAX_HORIZON + N_CV_SPLITS_PREDICT 
    if len(series_data_input) < min_len:
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in PREDICTION_HORIZONS}
    
    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS_PREDICT, test_size=MAX_HORIZON)
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS}
    
    last_fold_data = {}
    for fold, (train_idx, test_idx) in enumerate(tscv.split(series_data_input)):
        train_series = series_data_input.iloc[train_idx]
        context_unscaled = train_series.tail(CONTEXT_LENGTH).values
        if len(context_unscaled) < CONTEXT_LENGTH: continue
        
        context_scaled = scaler.transform(context_unscaled.reshape(-1, 1)).flatten()
        context_tensor = torch.tensor(context_scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            try:
                outputs = model_pipeline(x_enc=context_tensor, forecast_len=MAX_HORIZON)
                predictions_scaled = np.atleast_1d(outputs.forecast.cpu().squeeze().numpy())
            except Exception: continue
        
        ground_truth_unscaled = series_data_ground_truth.iloc[test_idx].values
        predictions_unscaled = scaler.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()

        for h in PREDICTION_HORIZONS:
            y_pred_h, y_true_h = predictions_unscaled[h-1], ground_truth_unscaled[h-1]
            accumulated_metrics[h]['maes'].append(np.abs(y_true_h - y_pred_h))
            accumulated_metrics[h]['rmses'].append((y_true_h - y_pred_h)**2)
            mase = calculate_mase(np.array([y_true_h]), np.array([y_pred_h]), train_series)
            if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)
        
        if fold == N_CV_SPLITS_PREDICT - 1:
            start_idx, end_idx = test_idx[0] - CONTEXT_LENGTH, test_idx[-1]
            last_fold_data = {'context': context_unscaled, 'ground_truth': ground_truth_unscaled, 'prediction': predictions_unscaled, 'date_index': df_input_data['Date'].iloc[start_idx:end_idx+1]}

    if last_fold_data and 'plot_path' in kwargs and 'csv_path' in kwargs:
        plot_and_save_forecast(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], last_fold_data['date_index'], kwargs['plot_title'], kwargs['plot_path'])
        save_prediction_results_to_csv(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], last_fold_data['date_index'], kwargs['csv_path'])
        print(f"      -> Gráfica y CSV de predicción del último fold guardados.")

    results = {}
    for h in PREDICTION_HORIZONS:
        results[h] = {'MAE': np.nanmean(accumulated_metrics[h]['maes']), 'RMSE': np.sqrt(np.nanmean(accumulated_metrics[h]['rmses'])), 'MASE': np.nanmean(accumulated_metrics[h]['mases'])}
    return results

# --- 3. Script Principal ---
if __name__ == '__main__':
    all_prediction_metrics_detailed = []
    script_start_time_pred = time.time()
    training_device = "cuda" if torch.cuda.is_available() else "cpu"
    prediction_device = "cpu"
    print(f"--- Predicción con MOMENT (Fine-Tuned) ---\nDispositivo de entrenamiento: {training_device.upper()} | Dispositivo de predicción: {prediction_device.upper()}")

    try:
        df_full_original_raw = pd.read_csv(ORIGINAL_DATA_FILE, parse_dates=['Date'])
        # Crear una versión limpia para usarla siempre como ground truth
        df_full_original_clean = df_full_original_raw.copy()
        df_full_original_clean[TARGET_VARIABLE] = df_full_original_clean[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()

        if TARGET_CITY_FILTER: cities_to_process = [TARGET_CITY_FILTER]
        else: cities_to_process = df_full_original_clean['City'].unique()
        print(f"\nATENCIÓN: Se procesarán las siguientes ciudades: {', '.join(cities_to_process)}")
    except FileNotFoundError:
        print(f"Error crítico: No se encontró el archivo original en {ORIGINAL_DATA_FILE}. Saliendo."); exit()

    finetuned_models, scalers = {}, {}
    for city in cities_to_process:
        print(f"\n--- Iniciando Fine-Tuning para la ciudad: {city} ---")
        df_city_clean = df_full_original_clean[df_full_original_clean['City'] == city].copy()
        if df_city_clean.empty: continue

        scaler = StandardScaler()
        train_size = int(len(df_city_clean) * FINETUNE_TRAIN_SPLIT_RATIO)
        train_data_scaled = scaler.fit_transform(df_city_clean[TARGET_VARIABLE].values[:train_size].reshape(-1, 1)).flatten()
        ft_dataset = MomentForecastingDataset(train_data_scaled, CONTEXT_LENGTH, MAX_HORIZON)
        ft_loader = DataLoader(ft_dataset, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0)
        
        if len(ft_loader) == 0: continue
        
        try:
            model_kwargs = {'task_name': 'forecasting', 'forecast_horizon': MAX_HORIZON, 'freeze_encoder': True, 'freeze_embedder': True}
            moment_predictor_ft = MOMENTPipeline.from_pretrained(MODEL_NAME, model_kwargs=model_kwargs)
            moment_predictor_ft.init(); moment_predictor_ft = moment_predictor_ft.to(training_device)
            
            city_safe_name = city.replace(" ", "_")
            model_save_path = os.path.join(MODELS_DIR, f"moment_forecaster_ft_{city_safe_name}.pth")
            scaler_save_path = os.path.join(SCALERS_DIR, f"scaler_{city_safe_name}.joblib")
            
            finetuned_model = train_moment_forecasting_model(moment_predictor_ft, ft_loader, training_device, FINETUNE_EPOCHS, TARGET_VARIABLE, model_save_path, MAX_HORIZON, LR)
            finetuned_models[city], scalers[city] = finetuned_model, scaler
            joblib.dump(scaler, scaler_save_path)
        except Exception as e:
            print(f"  Error crítico durante el fine-tuning para {city}: {e}"); traceback.print_exc()

    print("\n--- Iniciando Fase de Evaluación ---")
    method_name_baseline = "Baseline_Finetuned"
    print(f"\n--- Procesando {method_name_baseline} ---")
    for city in cities_to_process:
        if city in finetuned_models:
            print(f"  Ciudad: {city}")
            df_city_data = df_full_original_clean[df_full_original_clean['City'] == city].copy()
            model_to_eval, scaler_to_eval = finetuned_models[city].to(prediction_device), scalers[city]
            
            plot_title = f'MOMENT-FT Forecast in {city}\nImputation: Clean Data | Scenario: N/A'
            plot_path = os.path.join(PREDICTION_PLOTS_DIR, method_name_baseline, f'forecast_{city.replace(" ", "_")}.png')
            csv_path = os.path.join(PREDICTION_CSVS_DIR, method_name_baseline, f'forecast_{city.replace(" ", "_")}.csv')

            horizon_metrics = predict_and_evaluate_moment_cv(df_city_data, df_city_data, TARGET_VARIABLE, model_to_eval, prediction_device, scaler_to_eval, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
            for h, metrics in horizon_metrics.items():
                all_prediction_metrics_detailed.append({'Imputation_Method': method_name_baseline, 'City': city, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})

    # Datos Imputados
    print("\n--- Procesando Datos Imputados ---")
    imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
    for method_name in imputation_methods:
        imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
        if not os.path.exists(imputed_csvs_root): continue
        
        print(f"\n--- Método de Imputación: {method_name} ---")
        city_folders_to_process_paths = [os.path.join(imputed_csvs_root, c) for c in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, c))]
        
        for city_path in city_folders_to_process_paths:
            city_name = os.path.basename(city_path).replace("_", " ")
            if city_name not in cities_to_process or city_name not in finetuned_models: continue
            print(f"  Ciudad: {city_name}")

            model_to_eval, scaler_to_eval = finetuned_models[city_name].to(prediction_device), scalers[city_name]
            df_city_ground_truth = df_full_original_clean[df_full_original_clean['City'] == city_name].copy()

            for scenario_folder in os.listdir(city_path):
                scenario_path = os.path.join(city_path, scenario_folder)
                if not os.path.isdir(scenario_path): continue
                print(f"    Escenario: {scenario_folder}")
                
                files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
                if not files_to_process: continue
                
                for rep_file_path in files_to_process:
                    file_name = os.path.basename(rep_file_path)
                    rep_id = file_name
                    print(f"      -> Procesando archivo: {file_name}")

                    try:
                        df_rep = pd.read_csv(rep_file_path, parse_dates=['Date'])
                        if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                        if df_rep.empty: continue
                    except Exception as e: print(f"        Error cargando {file_name}: {e}"); continue
                    
                    city_safe_name = city_name.replace(" ","_")
                    safe_rep_id = rep_id.replace('.csv', '')
                    plot_title = f'MOMENT-FT Forecast in {city_name}\nImputation: {method_name} | Scenario: {scenario_folder}'
                    plot_dir = os.path.join(PREDICTION_PLOTS_DIR, method_name, city_safe_name, scenario_folder)
                    csv_dir = os.path.join(PREDICTION_CSVS_DIR, method_name, city_safe_name, scenario_folder)
                    plot_path = os.path.join(plot_dir, f'{safe_rep_id}.png')
                    csv_path = os.path.join(csv_dir, f'{safe_rep_id}.csv')
                    
                    horizon_metrics = predict_and_evaluate_moment_cv(df_rep, df_city_ground_truth, TARGET_VARIABLE, model_to_eval, prediction_device, scaler_to_eval, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
                    for h, metrics in horizon_metrics.items():
                        all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

    # --- 4. Guardado de Métricas ---
    if all_prediction_metrics_detailed:
        df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
        detailed_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.to_csv(detailed_path, index=False)
        print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")
        summary_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
        df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
        print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")
        global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
        global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
        global_summary.to_csv(global_summary_path, index=False)
        print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")
        final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
        final_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
        final_summary.to_csv(final_summary_path, index=False)
        print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
        print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary.sort_values(by='Final_Avg_MAE'))

    script_end_time = time.time()
    print(f"\nScript completado. Tiempo total: {(script_end_time - script_start_time_pred)/60:.2f} minutos.")

c:\Users\felix\anaconda3\envs\moment_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\felix\anaconda3\envs\moment_env\Lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


--- Predicción con MOMENT (Fine-Tuned) ---
Dispositivo de entrenamiento: CUDA | Dispositivo de predicción: CPU

ATENCIÓN: Se procesarán las siguientes ciudades: Brasilia, Cairo, Dubai, London, New York, Sydney

--- Iniciando Fine-Tuning para la ciudad: Brasilia ---
  -> Iniciando fine-tuning para 'AQI' en CUDA (1 épocas máx)...


    Epoch 1 Loss: 0.312677
  -> Fine-tuning completado. Mejor pérdida: 0.312677

--- Iniciando Fine-Tuning para la ciudad: Cairo ---
  -> Iniciando fine-tuning para 'AQI' en CUDA (1 épocas máx)...


    Epoch 1 Loss: 0.553856
  -> Fine-tuning completado. Mejor pérdida: 0.553856

--- Iniciando Fine-Tuning para la ciudad: Dubai ---
  -> Iniciando fine-tuning para 'AQI' en CUDA (1 épocas máx)...


    Epoch 1 Loss: 0.401110
  -> Fine-tuning completado. Mejor pérdida: 0.401110

--- Iniciando Fine-Tuning para la ciudad: London ---
  -> Iniciando fine-tuning para 'AQI' en CUDA (1 épocas máx)...


    Epoch 1 Loss: 0.729766
  -> Fine-tuning completado. Mejor pérdida: 0.729766

--- Iniciando Fine-Tuning para la ciudad: New York ---
  -> Iniciando fine-tuning para 'AQI' en CUDA (1 épocas máx)...


    Epoch 1 Loss: 0.624705
  -> Fine-tuning completado. Mejor pérdida: 0.624705

--- Iniciando Fine-Tuning para la ciudad: Sydney ---
  -> Iniciando fine-tuning para 'AQI' en CUDA (1 épocas máx)...


    Epoch 1 Loss: 0.761113
  -> Fine-tuning completado. Mejor pérdida: 0.761113

--- Iniciando Fase de Evaluación ---

--- Procesando Baseline_Finetuned ---
  Ciudad: Brasilia
      -> Gráfica y CSV de predicción del último fold guardados.
  Ciudad: Cairo
      -> Gráfica y CSV de predicción del último fold guardados.
  Ciudad: Dubai
      -> Gráfica y CSV de predicción del último fold guardados.
  Ciudad: London
      -> Gráfica y CSV de predicción del último fold guardados.
  Ciudad: New York
      -> Gráfica y CSV de predicción del último fold guardados.
  Ciudad: Sydney
      -> Gráfica y CSV de predicción del último fold guardados.

--- Procesando Datos Imputados ---

--- Método de Imputación: Linear Interpolation ---
  Ciudad: Brasilia
    Escenario: BlockRows_12h_20percent
      -> Procesando archivo: Brasilia_BlockRows_12h_20percent_rep1_seed2510790048_imputed_interpolacionLineal_FULL_plots.csv
      -> Gráfica y CSV de predicción del último fold guardados.
    Escenario: Block